# OHAYO - QLoRA Verification
Code cell 1-3: Input/moi truong. Cell 4: reload adapter va so sanh 20 ca (40 dau ra). Cell 5: resume 25 -> 26 dung 1 update. Cell 6: audit + ZIP. Khong train lai pilot 50.
Giu phien/Input/cache hien tai, GPU T4 va Internet. Neu mat phien, them ZIP pilot goc vao mot Dataset Private/Input; khong upload base-cache. Khong chay cung inference server.
Ket qua 20 ca chi la development, khong phai holdout. Format/citation ID/lexical overlap khong thay cham dung noi dung va bam nguon.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib, io, os, stat, sys
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/qlora-preparation')
EXPECTED = {'tools.bundle': '7fd1a989602f72951e3e7626f605d6509234183e04ef165f21e8eca35fc15b55', 'pilot.bundle': '2b44e35f7386b506955e9fad29541490c052129a8af6ae0eef7e4f0b7f7c0707'}
def obtain(name):
    candidates = []
    for path in INPUT.rglob(name):
        if path.is_file() and path.stat().st_size <= 64 * 1024**2:
            raw = path.read_bytes()
            if hashlib.sha256(raw).hexdigest() == EXPECTED[name]:
                candidates.append(raw)
    if not candidates:
        for path in INPUT.rglob('qlora-environment-input.zip'):
            with ZipFile(path) as archive:
                info = archive.getinfo(name)
                assert info.file_size <= 64 * 1024**2, 'INPUT_TOO_LARGE'
                raw = archive.read(info)
                if hashlib.sha256(raw).hexdigest() == EXPECTED[name]:
                    candidates.append(raw)
    assert len(candidates) == 1, f'{name}: can dung 1 Input khop checksum, tim thay {len(candidates)}'
    return candidates[0]
def unpack(raw, destination):
    destination = destination.resolve()
    with ZipFile(io.BytesIO(raw)) as archive:
        infos = archive.infolist()
        names = [entry.filename for entry in infos]
        assert len(infos) <= 200 and len(names) == len(set(n.casefold() for n in names)), 'ZIP_ENTRY_SET'
        assert sum(entry.file_size for entry in infos) <= 64 * 1024**2, 'ZIP_TOO_LARGE'
        assert archive.testzip() is None, 'ZIP_CRC'
        for entry in infos:
            name = entry.filename
            assert not entry.is_dir() and not stat.S_ISLNK(entry.external_attr >> 16), 'ZIP_MEMBER_TYPE'
            assert '\\' not in name and ':' not in name and not name.startswith('/'), 'ZIP_PATH'
            path = (destination / name).resolve()
            assert path.is_relative_to(destination) and path != destination, 'ZIP_TRAVERSAL'
            content = archive.read(entry)
            if path.exists():
                assert path.is_file() and path.read_bytes() == content, 'EXISTING_INPUT_DIFFERS'
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                with path.open('xb') as stream:
                    stream.write(content)
        assert {p.relative_to(destination).as_posix() for p in destination.rglob('*') if p.is_file()} == set(names), 'UNEXPECTED_EXTRACTED_FILES'
TOOLS = WORK / 'tools'
HANDOFF = WORK / 'handoff'
unpack(obtain('tools.bundle'), TOOLS)
unpack(obtain('pilot.bundle'), HANDOFF)
print('INPUTS_VERIFIED - code va pilot da duoc kiem checksum va giai nen.')


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib, json, os, shutil, subprocess
SINGLE_GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
VENV = Path('/kaggle/working/qlora-env')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(VENV)], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(VENV), 'install', '--disable-pip-version-check', 'pip==' + version('pip')], check=True)

# Corrected environment checker, copied into a new runtime tree, not locked Input.
SETUP_SOURCE = '"""GPU environment checks only: no Qwen weights, LoRA training or optimizer steps."""\nimport argparse\nimport gc\nfrom importlib.metadata import version\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom contextlib import redirect_stderr, redirect_stdout\n\nfrom .collator import CompletionCollator\nfrom .data import load_pilot, read_json, require\nfrom .runtime import environment, finish_manifest, output_guard, write_json\n\n\ndef check_hardware(torch, working):\n    require(torch.cuda.is_available(), \'CUDA_REQUIRED\')\n    require(torch.cuda.device_count() == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    require(torch.version.cuda is not None, \'CUDA_TORCH_REQUIRED_NOT_CPU_WHEEL\')\n    require(\'T4\' in torch.cuda.get_device_name(0), \'PROFILE_REQUIRES_T4\')\n    free, total = torch.cuda.mem_get_info(0)\n    require(free >= 12 * 1024 ** 3, \'GPU_BUSY_OR_INSUFFICIENT_FREE_VRAM\')\n    disk = shutil.disk_usage(working)\n    # This phase writes small reports/tensors only; base-cache budgeting belongs\n    # to the weight-loading phase and must not be reported as verified here.\n    require(disk.free >= 1024 ** 3, \'NEED_1_GIB_FREE_FOR_ENVIRONMENT_CHECK\')\n    return dict(device=torch.cuda.get_device_name(0), free_vram_bytes=free,\n                total_vram_bytes=total, free_working_disk_bytes=disk.free,\n                free_working_disk_gib=round(disk.free / 1024 ** 3, 3),\n                training_storage_check=\'PENDING_BEFORE_BASE_WEIGHT_LOADING\')\n\n\ndef probe_nf4(torch):\n    import bitsandbytes as bnb\n    # A random frozen layer tests CUDA kernels, not Qwen or an optimizer update.\n    torch.manual_seed(42)\n    with torch.cuda.device(0):\n        layer = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=torch.float16,\n                                  compress_statistics=True, quant_type=\'nf4\')\n        layer.requires_grad_(False)\n        layer = layer.to(\'cuda:0\')\n        sample = torch.randn(2, 64, device=\'cuda:0\', dtype=torch.float16, requires_grad=True)\n        output = layer(sample)\n        require(output.dtype == torch.float16 and torch.isfinite(output).all().item(), \'NF4_FP16_FORWARD\')\n        output.float().square().mean().backward()\n        require(sample.grad is not None and torch.isfinite(sample.grad).all().item()\n                and sample.grad.abs().sum().item() > 0, \'NF4_INPUT_GRADIENT\')\n        state = layer.weight.quant_state\n        require(layer.weight.device.type == \'cuda\' and state.quant_type == \'nf4\'\n                and state.nested, \'NF4_DOUBLE_QUANTIZATION\')\n        torch.cuda.synchronize(0)\n        del sample, output, layer\n    gc.collect()\n    torch.cuda.empty_cache()\n    return dict(nf4_cuda_forward=\'PASSED\', fp16_compute=\'PASSED\', input_backward=\'PASSED\',\n                double_quantization=True, optimizer_steps=0, qwen_weights_loaded=False)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', type=Path, required=True)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\',\n                      PYTHONDONTWRITEBYTECODE=\'1\', WANDB_DISABLED=\'true\')\n    config = read_json(args.config)\n    output = output_guard(args.output, [args.bundle, args.config])\n    output.mkdir(parents=True, exist_ok=False)\n    status, error = \'FAILED\', None\n    try:\n        with (output / \'setup-output.txt\').open(\'w\', encoding=\'utf-8\') as stream, \\\n             redirect_stdout(stream), redirect_stderr(stream):\n            import torch\n            from packaging.version import Version\n            write_json(output / \'environment.json\', environment())\n            require(Version(torch.__version__.split(\'+\')[0]) >= Version(\'2.6\'), \'TORCH_2_6_OR_NEWER_REQUIRED\')\n            for name, expected in config[\'dependencies\'].items():\n                require(version(name) == expected, \'DEPENDENCY_VERSION:\' + name)\n            hardware = check_hardware(torch, output)\n            from transformers import Trainer, TrainingArguments\n            from peft import LoraConfig, prepare_model_for_kbit_training\n            from accelerate import Accelerator\n            probe = probe_nf4(torch)\n            data, tokenizer, contract = load_pilot(args.bundle / \'data/pilot-v2\',\n                                                   args.bundle / \'tokenizer\', config)\n            collator = CompletionCollator()\n            for rows in data.values():\n                for offset in range(0, len(rows), 4):\n                    collator(rows[offset:offset + 4])\n            write_json(output / \'environment-check.json\', dict(\n                status=\'ENVIRONMENT_READY_FOR_SMOKE\', hardware=hardware, kernel_probe=probe,\n                records_checked=sum(map(len, data.values())), train=len(data[\'train\']),\n                validation=len(data[\'validation\']), qwen_weights_loaded=False,\n                qwen_training=\'NOT_RUN\', gpu_smoke=\'NOT_RUN\', gpu_pilot=\'NOT_RUN\'))\n            status = \'ENVIRONMENT_READY_FOR_SMOKE\'\n    except BaseException as exc:\n        error = type(exc).__name__\n        write_json(output / \'environment-check.json\', dict(status=\'FAILED\', error_type=error,\n                   error=str(exc), qwen_training=\'NOT_RUN\', gpu_smoke=\'NOT_RUN\', gpu_pilot=\'NOT_RUN\'))\n        raise\n    finally:\n        finish_manifest(output, status=status, profile=\'environment-only\', config=config, error_type=error)\n    print(status + \': \' + str(output))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n'
SETUP_SHA256 = 'cc9810b4414fddabf62757891e59f1daf22c57730bee1dd299d0da68c39f0d8a'
assert hashlib.sha256(SETUP_SOURCE.encode('utf-8')).hexdigest() == SETUP_SHA256
ORIGINAL_TOOLS = WORK / 'tools'
RUN_TOOLS = WORK / ('tools-environment-' + SETUP_SHA256[:12])
if not RUN_TOOLS.exists():
    shutil.copytree(ORIGINAL_TOOLS, RUN_TOOLS)
    (RUN_TOOLS / 'scripts/qlora/setup.py').write_bytes(SETUP_SOURCE.encode('utf-8'))
for original in ORIGINAL_TOOLS.rglob('*'):
    if original.is_file():
        relative = original.relative_to(ORIGINAL_TOOLS)
        actual = RUN_TOOLS / relative
        if relative.as_posix() == 'scripts/qlora/setup.py':
            assert hashlib.sha256(actual.read_bytes()).hexdigest() == SETUP_SHA256
        else:
            assert actual.read_bytes() == original.read_bytes(), 'RUNTIME_COPY_DIFFERS'
assert {p.relative_to(RUN_TOOLS).as_posix() for p in RUN_TOOLS.rglob('*') if p.is_file()} == {p.relative_to(ORIGINAL_TOOLS).as_posix() for p in ORIGINAL_TOOLS.rglob('*') if p.is_file()}
TOOLS = RUN_TOOLS
REPORT = WORK / 'reports' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
command = [sys.executable, '-B', '-m', 'scripts.qlora.kaggle', '--bundle', str(HANDOFF), '--profile', 'smoke', '--environment-only', '--output', str(REPORT)]
subprocess.run(command, cwd=TOOLS, env=SINGLE_GPU_ENV, check=True)


In [ ]:
import json
result = json.loads((REPORT / 'environment-check.json').read_text())
print(json.dumps(result, ensure_ascii=False, indent=2))
print('Bao cao:', REPORT)
assert result['status'] == 'ENVIRONMENT_READY_FOR_SMOKE', 'Moi truong chua dat; gui loi de kiem tra.'
print('Moi truong da dat. Chua chay smoke/pilot Qwen. Gui output cell nay de xac nhan buoc tiep theo.')


In [ ]:
# Reload the saved adapter and compare exactly 20 locked development cases.
import os, time
assert json.loads((REPORT / 'environment-check.json').read_text())['status'] == 'ENVIRONMENT_READY_FOR_SMOKE'
for name in ('SMOKE_PROCESS', 'PILOT_PROCESS', 'VERIFY_PROCESS'):
    previous = globals().get(name)
    if previous is not None and previous.poll() is None:
        raise RuntimeError('A GPU process is already running.')
VERIFY_SOURCES = {'data.py': '"""Authenticate the locked pilot and recompute its tokenizer boundaries on CPU."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport unicodedata\n\n\ndef require(condition, code):\n    if not condition:\n        raise ValueError(code)\n\n\ndef read_json(path):\n    return loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef loads(text):\n    def pairs(items):\n        result = {}\n        for key, value in items:\n            require(key not in result, \'DUPLICATE_JSON_KEY\')\n            result[key] = value\n        return result\n    def invalid(value):\n        raise ValueError(\'NONFINITE_JSON\')\n    return json.loads(text, object_pairs_hook=pairs, parse_constant=invalid)\n\n\ndef canonical(value):\n    def normalize(item):\n        if isinstance(item, str):\n            return unicodedata.normalize(\'NFC\', item)\n        if isinstance(item, list):\n            return [normalize(v) for v in item]\n        if isinstance(item, dict):\n            return {k: normalize(v) for k, v in item.items()}\n        return item\n    return json.dumps(normalize(value), ensure_ascii=False, sort_keys=True,\n                      separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef sha_file(path):\n    sha = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            sha.update(block)\n    return sha.hexdigest()\n\n\ndef within(root, name):\n    root = Path(root).resolve()\n    require(isinstance(name, str) and \'\\\\\' not in name and \':\' not in name,\n            \'UNSAFE_MEMBER_PATH\')\n    path = (root / name).resolve()\n    require(path.is_relative_to(root) and path != root, \'MEMBER_OUTSIDE_ROOT\')\n    return path\n\n\ndef read_rows(path):\n    rows = []\n    for line in Path(path).read_text(encoding=\'utf-8\').splitlines():\n        require(bool(line.strip()), \'EMPTY_JSONL_ROW\')\n        row = loads(line)\n        require(isinstance(row, dict), \'INVALID_JSONL_ROW\')\n        rows.append(row)\n    return rows\n\n\ndef tokenize_payload(payload, tokenizer, limits):\n    prompt = tokenizer.apply_chat_template(payload[\'prompt\'], tokenize=True,\n                                            add_generation_prompt=True)\n    full = tokenizer.apply_chat_template(payload[\'prompt\'] + payload[\'completion\'],\n                                          tokenize=True, add_generation_prompt=False)\n    require(isinstance(full, list) and isinstance(prompt, list)\n            and all(type(n) is int for n in full + prompt), \'TOKENIZER_IDS\')\n    require(full[:len(prompt)] == prompt, \'CHAT_TEMPLATE_PREFIX\')\n    completion = full[len(prompt):]\n    require(completion.count(tokenizer.eos_token_id) == 1, \'COMPLETION_EOS\')\n    eos = len(prompt) + completion.index(tokenizer.eos_token_id)\n    tail = full[eos + 1:]\n    require(eos > len(prompt), \'EMPTY_COMPLETION\')\n    require(not tail or tail == tokenizer.encode(\'\\n\', add_special_tokens=False), \'POST_EOS_TEXT\')\n    require(len(prompt) <= limits[\'max_prompt_tokens\']\n            and len(completion) <= limits[\'max_completion_tokens\']\n            and len(full) <= limits[\'max_sequence_tokens\'], \'TOKEN_BUDGET\')\n    return dict(input_ids=full, attention_mask=[1] * len(full),\n                labels=[-100] * len(prompt) + full[len(prompt):eos + 1] + [-100] * len(tail),\n                prompt_tokens=len(prompt), completion_tokens=len(completion),\n                sequence_tokens=len(full), eos_position=eos,\n                supervised_tokens=eos + 1 - len(prompt))\n\n\ndef authenticate(folder, config):\n    folder = Path(folder).resolve()\n    manifest_path = folder / \'manifest.json\'\n    require(sha_file(manifest_path) == config[\'data_manifest_sha256\'], \'MANIFEST_HASH\')\n    manifest = read_json(manifest_path)\n    for name, info in manifest[\'files\'].items():\n        path = within(folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'DATA_INTEGRITY:\' + name)\n    contract = read_json(folder / \'training-contract.json\')\n    require(digest(contract) == config[\'contract_sha256\'] == manifest[\'contract_sha256\'],\n            \'CONTRACT_HASH\')\n    require(config[\'model_id\'] == contract[\'model\'][\'model_id\']\n            and config[\'revision\'] == contract[\'model\'][\'revision\']\n            == contract[\'model\'][\'tokenizer_revision\'], \'MODEL_IDENTITY\')\n    selection = read_json(folder / \'selection.json\')\n    require(set(selection[\'splits\']) == {\'train\', \'validation\'}, \'PILOT_SPLITS\')\n    ids = selection[\'splits\'][\'train\'] + selection[\'splits\'][\'validation\']\n    require(len(ids) == len(set(ids)) and not set(ids) & set(selection[\'excluded_holdout_ids\']),\n            \'DUPLICATE_OR_HOLDOUT_ID\')\n    require(manifest[\'counts\'] == {\'train\': 128, \'validation\': 32}\n            and manifest[\'holdout_payload_included\'] is False, \'PILOT_COUNTS\')\n    return contract, manifest, selection\n\n\ndef load_pilot(folder, tokenizer_folder, config):\n    from importlib.metadata import version\n    from transformers import AutoTokenizer\n    folder = Path(folder).resolve()\n    contract, manifest, selection = authenticate(folder, config)\n    reference = read_json(folder / \'tokenizer-reference.json\')\n    for name, expected in reference[\'packages\'].items():\n        require(version(name) == expected, \'TOKENIZER_LIBRARY_VERSION:\' + name)\n    for name, info in reference[\'files\'].items():\n        path = within(tokenizer_folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'TOKENIZER_ASSET:\' + name)\n    tokenizer = AutoTokenizer.from_pretrained(tokenizer_folder, local_files_only=True,\n                                               trust_remote_code=False)\n    require(tokenizer.eos_token_id == contract[\'model\'][\'eos_token_id\']\n            and tokenizer.pad_token_id == contract[\'model\'][\'pad_token_id\']\n            and hashlib.sha256(tokenizer.chat_template.encode()).hexdigest()\n            == contract[\'model\'][\'chat_template_sha256\'], \'TOKENIZER_IDENTITY\')\n    result = {}\n    for split in (\'train\', \'validation\'):\n        tokens = read_rows(folder / (\'tokenized-\' + split + \'.jsonl\'))\n        payloads = read_rows(folder / (split + \'.jsonl\'))\n        metadata = read_rows(folder / (split + \'.metadata.jsonl\'))\n        require(len(tokens) == len(payloads) == len(metadata) == manifest[\'counts\'][split],\n                \'SPLIT_COUNT\')\n        result[split] = []\n        for index, (row, payload, meta) in enumerate(zip(tokens, payloads, metadata)):\n            rid = selection[\'splits\'][split][index]\n            require(row[\'record_id\'] == meta[\'source_record_id\'] == rid\n                    and meta[\'split\'] == split and meta[\'row_index\'] == index, \'ROW_BINDING\')\n            require(digest(payload) == meta[\'training_payload_sha256\'], \'PAYLOAD_HASH\')\n            require(row == dict(record_id=rid, **tokenize_payload(payload, tokenizer,\n                                                                contract[\'token_limits\'])),\n                    \'TOKEN_LABEL_RECOMPUTATION\')\n            # Do not allow IDs or provenance to reach model.forward().\n            result[split].append({k: row[k] for k in (\'input_ids\', \'attention_mask\', \'labels\')})\n    return result, tokenizer, contract\n', 'collator.py': '"""Strict right padding for already-audited, unshifted completion-only labels."""\nfrom dataclasses import dataclass\nfrom .data import require\n\n\n@dataclass(frozen=True)\nclass CompletionCollator:\n    pad_token_id: int = 151643\n    eos_token_id: int = 151645\n    max_sequence_tokens: int = 2048\n    pad_to_multiple_of: int = 8\n\n    def pad(self, features):\n        require(bool(features), \'EMPTY_BATCH\')\n        require(type(self.pad_to_multiple_of) is int and self.pad_to_multiple_of > 0,\n                \'PADDING_MULTIPLE\')\n        batch = {k: [] for k in (\'input_ids\', \'attention_mask\', \'labels\')}\n        for row in features:\n            require(set(row) == set(batch), \'MODEL_INPUT_FIELDS\')\n            ids, mask, labels = (row[k] for k in batch)\n            require(all(isinstance(v, list) for v in (ids, mask, labels))\n                    and len(ids) == len(mask) == len(labels) > 0, \'TOKEN_SHAPES\')\n            require(all(type(n) is int and n >= 0 for n in ids)\n                    and all(type(n) is int and n == 1 for n in mask)\n                    and all(type(n) is int for n in labels), \'TOKEN_TYPES_OR_PREPADDING\')\n            positions = [i for i, n in enumerate(labels) if n != -100]\n            require(len(positions) >= 2 and positions[0] > 0\n                    and positions == list(range(positions[0], positions[-1] + 1)),\n                    \'COMPLETION_MASK\')\n            require(all(labels[i] == ids[i] for i in positions)\n                    and labels[positions[-1]] == self.eos_token_id\n                    and sum(labels[i] == self.eos_token_id for i in positions) == 1,\n                    \'LABEL_IDS_OR_EOS\')\n        size = max(len(row[\'input_ids\']) for row in features)\n        size = ((size + self.pad_to_multiple_of - 1) // self.pad_to_multiple_of) * self.pad_to_multiple_of\n        require(size <= self.max_sequence_tokens, \'PADDED_TOKEN_LIMIT\')\n        for row in features:\n            missing = size - len(row[\'input_ids\'])\n            for key, padding in ((\'input_ids\', self.pad_token_id), (\'attention_mask\', 0), (\'labels\', -100)):\n                batch[key].append(row[key][:] + [padding] * missing)\n        return batch\n\n    def __call__(self, features):\n        import torch\n        return {key: torch.tensor(values, dtype=torch.long) for key, values in self.pad(features).items()}\n', 'runtime.py': '"""Trainer setup, update evidence and authenticated resume, independent of serving."""\nfrom datetime import datetime, timezone\nfrom importlib.metadata import distributions, version\nimport json\nimport math\nimport platform\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom .data import digest, read_json, require, sha_file, within\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,\n                                    allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n\n\ndef inventory(folder, excluded=()):\n    folder = Path(folder).resolve()\n    result = {}\n    for path in sorted(folder.rglob(\'*\')):\n        require(not path.is_symlink(), \'ARTIFACT_SYMLINK\')\n        if path.is_file() and path.relative_to(folder).as_posix() not in excluded:\n            result[path.relative_to(folder).as_posix()] = {\n                \'sha256\': sha_file(path), \'size_bytes\': path.stat().st_size}\n    return result\n\n\ndef verify_inventory(folder, files):\n    for name, info in files.items():\n        path = within(folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'ARTIFACT_INTEGRITY:\' + name)\n\n\ndef validate_recipe(config, profile):\n    require(config[\'schema_version\'] == \'ohayo-qlora-recipe-v1\', \'RECIPE_VERSION\')\n    require(profile in (\'smoke\', \'pilot\'), \'PROFILE\')\n    require(config[\'profiles\'] == {\n        \'smoke\': {\'max_steps\': 2, \'save_steps\': 2, \'eval_steps\': 2},\n        \'pilot\': {\'max_steps\': 50, \'save_steps\': 25, \'eval_steps\': 25}}, \'STEP_PROFILES\')\n    train = config[\'training\']\n    require(train[\'per_device_train_batch_size\'] == train[\'per_device_eval_batch_size\'] == 1\n            and train[\'gradient_accumulation_steps\'] == 4, \'BATCH_RECIPE\')\n    require(all(isinstance(train[k], (float, int)) and not isinstance(train[k], bool)\n                and math.isfinite(train[k]) and train[k] > 0\n                for k in (\'learning_rate\', \'max_grad_norm\'))\n            and 0 <= train[\'warmup_ratio\'] <= 1, \'OPTIMIZER_RECIPE\')\n    require(config[\'seed\'] == 42 and config[\'lora\'] == {\n        \'r\': 8, \'lora_alpha\': 16, \'lora_dropout\': 0.05, \'bias\': \'none\',\n        \'target_modules\': \'all-linear\', \'task_type\': \'CAUSAL_LM\'}, \'LORA_RECIPE\')\n\n\ndef output_guard(output, inputs):\n    output = Path(output).resolve()\n    require(not output.exists(), \'OUTPUT_ALREADY_EXISTS\')\n    for value in inputs:\n        value = Path(value).resolve()\n        require(not output.is_relative_to(value) and not value.is_relative_to(output),\n                \'OUTPUT_OVERLAPS_INPUT\')\n    return output\n\n\ndef training_arguments(config, profile, output, *, cpu_test=False):\n    from transformers import TrainingArguments\n    values = dict(config[\'training\'])\n    args = TrainingArguments(\n        output_dir=str(output), **values, **config[\'profiles\'][profile],\n        seed=config[\'seed\'], data_seed=config[\'seed\'],\n        eval_strategy=\'steps\', save_strategy=\'steps\', logging_strategy=\'steps\', logging_steps=1,\n        save_total_limit=2, save_only_model=False, load_best_model_at_end=False,\n        fp16=not cpu_test, bf16=False, use_cpu=cpu_test,\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={\'use_reentrant\': False},\n        report_to=\'none\', remove_unused_columns=False, label_names=[\'labels\'],\n        dataloader_num_workers=0, dataloader_pin_memory=not cpu_test,\n        disable_tqdm=True, save_safetensors=True, logging_nan_inf_filter=False)\n    if not cpu_test:\n        require(args.n_gpu == 1 and args.world_size == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    return args\n\n\ndef adapter_parameters(model):\n    selected = {}\n    for name, param in model.named_parameters():\n        if param.requires_grad:\n            require(\'lora_\' in name, \'BASE_PARAMETER_TRAINABLE:\' + name)\n            selected[name] = param\n    require(bool(selected), \'NO_TRAINABLE_ADAPTER\')\n    return selected\n\n\ndef parameter_report(model):\n    selected = adapter_parameters(model)\n    return dict(trainable_parameters=sum(p.numel() for p in selected.values()),\n                total_parameter_elements=sum(p.numel() for p in model.parameters()),\n                trainable_names=sorted(selected), base_frozen=True,\n                note=\'Quantized parameter elements are storage counts, not dense base parameter counts.\')\n\n\ndef resume_checkpoint(checkpoint, expected_manifest_sha256, config, stop_after_step):\n    checkpoint = Path(checkpoint).resolve()\n    parent = checkpoint.parent\n    require(expected_manifest_sha256 is not None, \'RESUME_RECEIPT_REQUIRED\')\n    require(sha_file(parent / \'run-manifest.json\') == expected_manifest_sha256,\n            \'RESUME_MANIFEST_HASH\')\n    report = read_json(parent / \'run-manifest.json\')\n    require(report[\'status\'] == \'TRAINING_COMPLETED\' and report[\'profile\'] == \'pilot\'\n            and report[\'config_sha256\'] == digest(config), \'RESUME_RUN_IDENTITY\')\n    verify_inventory(parent, report[\'files\'])\n    require(set(inventory(parent, (\'run-manifest.json\', \'SHA256SUMS.txt\')))\n            == set(report[\'files\']), \'RESUME_EXTRA_FILES\')\n    code = {p.name: sha_file(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}\n    require(report[\'source_files_sha256\'] == code, \'RESUME_TRAINER_CODE_CHANGED\')\n    require(checkpoint.name == \'checkpoint-25\' and checkpoint.is_dir(), \'RESUME_STEP_25_REQUIRED\')\n    binding = read_json(checkpoint / \'binding.json\')\n    require(binding[\'config_sha256\'] == digest(config)\n            and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\'], \'RESUME_BINDING\')\n    require(read_json(checkpoint / \'trainer_state.json\')[\'global_step\'] == 25,\n            \'RESUME_STATE_STEP\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\'}\n    require(required <= {p.name for p in checkpoint.iterdir()}, \'INCOMPLETE_RESUME_STATE\')\n    require(stop_after_step == 26, \'RESUME_STOP_26_REQUIRED\')\n    return checkpoint\n\n\ndef make_callbacks(output, config, *, stop_after_step=None):\n    import torch\n    from transformers import TrainerCallback\n\n    class EvidenceCallback(TrainerCallback):\n        trainer = None\n\n        def __init__(self):\n            self.before = {}\n            self.applied = 0\n            self.skipped = 0\n            self.gradient_checks = 0\n            self.started = None\n            self.start_step = None\n\n        def on_train_begin(self, args, state, control, model=None, **kwargs):\n            self.started = time.monotonic()\n            self.start_step = state.global_step\n            self.before = {n: p.detach().float().cpu().clone()\n                           for n, p in adapter_parameters(model).items()}\n\n        def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):\n            gradients = [p.grad for p in adapter_parameters(model).values() if p.grad is not None]\n            require(bool(gradients), \'NO_ADAPTER_GRADIENT\')\n            require(all(torch.isfinite(g).all().item() for g in gradients), \'NONFINITE_GRADIENT\')\n            self.gradient_checks += 1\n\n        def on_step_end(self, args, state, control, **kwargs):\n            skipped = bool(self.trainer.accelerator.optimizer_step_was_skipped)\n            self.skipped += int(skipped)\n            self.applied += int(not skipped)\n            event = dict(step=state.global_step, update_applied=not skipped,\n                         elapsed_seconds=time.monotonic() - self.started)\n            with (output / \'optimizer-events.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(event, allow_nan=False) + \'\\n\')\n            write_json(output / \'phase.json\', dict(phase=\'TRAINING_AND_VALIDATION\', **event))\n            if stop_after_step is not None and state.global_step >= stop_after_step:\n                control.should_training_stop = True\n                control.should_save = True\n                control.should_evaluate = True\n            return control\n\n        def on_log(self, args, state, control, logs=None, **kwargs):\n            logs = dict(logs or {})\n            for value in logs.values():\n                if isinstance(value, (int, float)):\n                    require(math.isfinite(value), \'NONFINITE_TRAINING_LOG\')\n            with (output / \'train-metrics.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(dict(step=state.global_step, **logs), allow_nan=False) + \'\\n\')\n\n        def on_save(self, args, state, control, **kwargs):\n            checkpoint = output / (\'checkpoint-\' + str(state.global_step))\n            write_json(checkpoint / \'binding.json\', {\n                \'config_sha256\': digest(config), \'data_manifest_sha256\': config[\'data_manifest_sha256\'],\n                \'global_step\': state.global_step})\n\n        def report(self, model, global_step):\n            after = adapter_parameters(model)\n            require(set(after) == set(self.before), \'ADAPTER_PARAMETERS_CHANGED\')\n            squared = sum((p.detach().float().cpu() - self.before[n]).double().square().sum().item()\n                          for n, p in after.items())\n            return dict(start_global_step=self.start_step, global_step=global_step,\n                        attempted_updates=self.applied + self.skipped,\n                        applied_updates=self.applied, scaler_skipped_updates=self.skipped,\n                        finite_gradient_checks=self.gradient_checks,\n                        adapter_delta_l2=math.sqrt(squared), base_frozen=True)\n\n    return EvidenceCallback()\n\n\ndef fit(model, data, collator, args, output, config, *, resume=None, stop_after_step=None):\n    from transformers import Trainer\n    if not args.use_cpu:\n        require(args.n_gpu == 1 and args.world_size == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    evidence = make_callbacks(output, config, stop_after_step=stop_after_step)\n    trainer = Trainer(model=model, args=args, train_dataset=data[\'train\'],\n                      eval_dataset=data[\'validation\'], data_collator=collator,\n                      callbacks=[evidence])\n    evidence.trainer = trainer\n    try:\n        trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    except BaseException as exc:\n        # A CUDA failure can poison the context. Do not copy tensors back to CPU\n        # while handling it, or replace the original exception with a second one.\n        failure = dict(error_type=type(exc).__name__, error=str(exc),\n                       global_step=trainer.state.global_step,\n                       observed_applied_updates=evidence.applied,\n                       observed_skipped_updates=evidence.skipped,\n                       adapter_delta_l2=None, tensor_measurement=\'NOT_ATTEMPTED_ON_FAILED_RUN\')\n        try:\n            write_json(output / \'training-failure.json\', failure)\n        except OSError:\n            pass\n        raise\n    update = evidence.report(model, trainer.state.global_step)\n    write_json(output / \'adapter-update-report.json\', update)\n    target = stop_after_step or args.max_steps\n    require(trainer.state.global_step == target and update[\'applied_updates\'] > 0\n            and update[\'scaler_skipped_updates\'] == 0\n            and update[\'attempted_updates\'] == target - update[\'start_global_step\']\n            and update[\'finite_gradient_checks\'] == update[\'attempted_updates\']\n            and math.isfinite(update[\'adapter_delta_l2\']) and update[\'adapter_delta_l2\'] > 0,\n            \'OPTIMIZER_UPDATE_PROOF_FAILED\')\n    metrics = next((entry for entry in reversed(trainer.state.log_history)\n                    if entry.get(\'step\') == trainer.state.global_step and \'eval_loss\' in entry), None)\n    if metrics is None:\n        metrics = trainer.evaluate()\n    require(math.isfinite(metrics[\'eval_loss\']), \'NONFINITE_EVAL_LOSS\')\n    write_json(output / \'eval-metrics.json\', metrics)\n    trainer.save_model(str(output / \'final-adapter\'))\n    trainer.save_state()\n    return trainer, update\n\n\ndef environment():\n    import torch\n    return dict(python=sys.version, platform=platform.platform(), torch=torch.__version__,\n                cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(),\n                gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                dependencies={d.metadata[\'Name\']: d.version for d in distributions()},\n                recorded_at=datetime.now(timezone.utc).isoformat())\n\n\ndef load_training_model(config, *, allow_download=False, cache_dir=None):\n    import torch\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    require(torch.cuda.is_available(), \'CUDA_REQUIRED\')\n    require(torch.cuda.device_count() == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    require(\'T4\' in torch.cuda.get_device_name(0), \'PROFILE_REQUIRES_T4\')\n    require(not torch.distributed.is_initialized(), \'SINGLE_PROCESS_ONLY\')\n    free, total = torch.cuda.mem_get_info(0)\n    require(free >= 12 * 1024 ** 3, \'GPU_BUSY_OR_INSUFFICIENT_FREE_VRAM\')\n    for name, expected in config[\'dependencies\'].items():\n        require(version(name) == expected, \'DEPENDENCY_VERSION:\' + name)\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                      bnb_4bit_use_double_quant=True,\n                                      bnb_4bit_compute_dtype=torch.float16)\n    model = AutoModelForCausalLM.from_pretrained(\n        config[\'model_id\'], revision=config[\'revision\'], trust_remote_code=False,\n        local_files_only=not allow_download, quantization_config=quantization,\n        torch_dtype=torch.float16, device_map={\'\': 0}, attn_implementation=\'sdpa\',\n        use_safetensors=True, cache_dir=str(cache_dir) if cache_dir is not None else None)\n    require(model.config._commit_hash == config[\'revision\'], \'BASE_REVISION_MISMATCH\')\n    require(getattr(model, \'is_loaded_in_4bit\', False), \'BASE_NOT_4BIT\')\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,\n                                            gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    model = get_peft_model(model, LoraConfig(**config[\'lora\'], revision=config[\'revision\']))\n    adapter_parameters(model)\n    return model\n\n\ndef finish_manifest(output, *, status, profile, config, error_type=None):\n    source = Path(__file__).parent\n    code = {p.name: sha_file(p) for p in sorted(source.glob(\'*.py\'))}\n    try:\n        git = subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], capture_output=True, text=True,\n                             check=True, cwd=source).stdout.strip()\n    except (OSError, subprocess.SubprocessError):\n        git = None\n    manifest = dict(status=status, profile=profile, config_sha256=digest(config),\n                    data_manifest_sha256=config[\'data_manifest_sha256\'],\n                    source_files_sha256=code, git_head=git,\n                    note=\'Source file hashes identify uncommitted code; git_head alone does not.\',\n                    error_type=error_type,\n                    files=inventory(output, (\'run-manifest.json\', \'SHA256SUMS.txt\')))\n    write_json(output / \'run-manifest.json\', manifest)\n    files = inventory(output, (\'SHA256SUMS.txt\',))\n    (output / \'SHA256SUMS.txt\').write_text(\'\'.join(\n        info[\'sha256\'] + \'  \' + name + \'\\n\' for name, info in files.items()), encoding=\'utf-8\')\n', 'train.py': '"""CLI: CPU preflight, separate T4 smoke/pilot runs, and authenticated resume."""\nimport argparse\nfrom contextlib import redirect_stderr, redirect_stdout\nimport os\nfrom pathlib import Path\nimport sys\n\nfrom .collator import CompletionCollator\nfrom .data import load_pilot, read_json, require\nfrom .runtime import (environment, finish_manifest, fit, load_training_model, output_guard,\n                      parameter_report, resume_checkpoint, training_arguments, validate_recipe, write_json)\nfrom .storage import check_training_storage\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--bundle\', type=Path, required=True, help=\'Extracted immutable handoff root\')\n    parser.add_argument(\'--profile\', choices=(\'smoke\', \'pilot\'), required=True)\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New private run directory\')\n    parser.add_argument(\'--preflight-only\', action=\'store_true\')\n    parser.add_argument(\'--allow-model-download\', action=\'store_true\')\n    parser.add_argument(\'--cache-dir\', type=Path)\n    parser.add_argument(\'--resume-from-checkpoint\', type=Path)\n    parser.add_argument(\'--resume-manifest-sha256\')\n    parser.add_argument(\'--stop-after-step\', type=int)\n    args = parser.parse_args(argv)\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\', WANDB_DISABLED=\'true\')\n    os.environ.setdefault(\'CUDA_VISIBLE_DEVICES\', \'0\')\n    config = read_json(args.config)\n    validate_recipe(config, args.profile)\n    inputs = [args.bundle, args.config]\n    if args.resume_from_checkpoint:\n        inputs.append(args.resume_from_checkpoint.parent)\n    output = output_guard(args.output, inputs)\n    require(args.resume_from_checkpoint is not None or (args.stop_after_step is None\n            and args.resume_manifest_sha256 is None), \'RESUME_ARGUMENTS_REQUIRED\')\n    require(not args.preflight_only or args.resume_from_checkpoint is None, \'PREFLIGHT_NOT_RESUME\')\n    resume = None\n    if args.resume_from_checkpoint:\n        require(args.profile == \'pilot\', \'RESUME_PILOT_ONLY\')\n        resume = resume_checkpoint(args.resume_from_checkpoint, args.resume_manifest_sha256,\n                                    config, args.stop_after_step)\n    output.mkdir(parents=True, exist_ok=False)\n    status, error = \'FAILED\', None\n    def phase(name):\n        write_json(output / \'phase.json\', {\'phase\': name, \'profile\': args.profile})\n    try:\n        phase(\'VALIDATING_DATA\')\n        with (output / \'stdout.txt\').open(\'w\', encoding=\'utf-8\') as stdout, \\\n             (output / \'stderr.txt\').open(\'w\', encoding=\'utf-8\') as stderr, \\\n             redirect_stdout(stdout), redirect_stderr(stderr):\n            import torch\n            from transformers import set_seed\n            set_seed(config[\'seed\'])\n            write_json(output / \'resolved-config.json\', config)\n            source = output / \'source/qlora\'\n            source.mkdir(parents=True)\n            for code in Path(__file__).parent.glob(\'*.py\'):\n                (source / code.name).write_bytes(code.read_bytes())\n            actual_environment = environment()\n            write_json(output / \'environment.json\', actual_environment)\n            (output / \'requirements-resolved.txt\').write_text(\'\'.join(\n                name + \'==\' + ver + \'\\n\' for name, ver in sorted(\n                    actual_environment[\'dependencies\'].items())), encoding=\'utf-8\')\n            write_json(output / \'data-reference.json\', {\n                \'manifest_sha256\': config[\'data_manifest_sha256\'],\n                \'contract_sha256\': config[\'contract_sha256\'],\n                \'train\': 128, \'validation\': 32, \'holdout\': 0,\n                \'resume_checkpoint\': str(resume) if resume else None,\n                \'resume_manifest_sha256\': args.resume_manifest_sha256})\n            data, tokenizer, contract = load_pilot(args.bundle / \'data/pilot-v2\',\n                                                   args.bundle / \'tokenizer\', config)\n            collator = CompletionCollator()\n            shapes = []\n            for split, rows in data.items():\n                for start in range(0, len(rows), 4):\n                    batch = collator(rows[start:start + 4])\n                    shapes.append(list(batch[\'input_ids\'].shape))\n            write_json(output / \'preflight.json\', {\n                \'passed\': True, \'records_checked\': sum(map(len, data.values())),\n                \'batch_shapes\': shapes, \'tensor_dtype\': \'torch.int64\',\n                \'tokenizer_recomputed\': True, \'model_weights_loaded\': False,\n                \'training\': \'NOT_RUN\'})\n            if args.preflight_only:\n                status = \'PREFLIGHT_PASSED\'\n            else:\n                from huggingface_hub.constants import HF_HUB_CACHE\n                cache = args.cache_dir or Path(HF_HUB_CACHE)\n                phase(\'CHECKING_TRAINING_STORAGE\')\n                storage = check_training_storage(config, cache, output,\n                                                   allow_download=args.allow_model_download)\n                write_json(output / \'storage-check.json\', storage)\n                require(storage[\'passed\'], \'INSUFFICIENT_DISK_FOR_PINNED_BASE_AND_RUN\')\n                phase(\'LOADING_PINNED_BASE\')\n                model = load_training_model(config, allow_download=args.allow_model_download,\n                                             cache_dir=cache)\n                base = model.get_base_model()\n                quantization = base.config.quantization_config\n                write_json(output / \'model-load-report.json\', {\n                    \'model_id\': config[\'model_id\'], \'revision\': base.config._commit_hash,\n                    \'model_class\': type(base).__name__, \'is_loaded_in_4bit\': base.is_loaded_in_4bit,\n                    \'quantization\': quantization.to_dict() if hasattr(quantization, \'to_dict\') else quantization,\n                    \'use_cache\': base.config.use_cache})\n                torch.cuda.reset_peak_memory_stats(0)\n                write_json(output / \'trainable-parameters.json\', parameter_report(model))\n                train_args = training_arguments(config, args.profile, output)\n                write_json(output / \'training-arguments.json\', train_args.to_dict())\n                write_json(output / \'training-device.json\', {\n                    \'visible_gpu_count\': train_args.n_gpu, \'world_size\': train_args.world_size,\n                    \'cuda_visible_devices\': os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n                    \'mode\': \'SINGLE_GPU_NO_DATAPARALLEL\'})\n                phase(\'TRAINING_AND_VALIDATION\')\n                _, update = fit(model, data, collator, train_args, output, config,\n                                 resume=resume, stop_after_step=args.stop_after_step)\n                phase(\'SAVING_ARTIFACTS\')\n                write_json(output / \'vram.json\', {\n                    \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(0),\n                    \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(0)})\n                tokenizer.save_pretrained(output / \'tokenizer\')\n                write_json(output / \'tokenizer-reference.json\', read_json(\n                    args.bundle / \'data/pilot-v2/tokenizer-reference.json\'))\n                status = \'TRAINING_COMPLETED\'\n            phase(status)\n    except BaseException as exc:\n        error = type(exc).__name__\n        phase(\'FAILED\')\n        # Do not print payloads or secrets into public console output.\n        with (output / \'stderr.txt\').open(\'a\', encoding=\'utf-8\') as stream:\n            stream.write(error + \': \' + str(exc) + \'\\n\')\n        raise\n    finally:\n        finish_manifest(output, status=status, profile=args.profile, config=config, error_type=error)\n    print(status + \': \' + str(output))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'storage.py': '"""Budget pinned safetensors downloads and run artifacts before loading any weights."""\nfrom pathlib import Path\nimport re\nimport shutil\n\nfrom .data import read_json, require\n\nGIB = 1024 ** 3\n\n\ndef budget(missing_bytes, cache_free, output_free, same_volume):\n    require(type(missing_bytes) is int and missing_bytes >= 0, \'INVALID_WEIGHT_SIZE\')\n    cache_required = missing_bytes + (GIB if missing_bytes else 0)\n    output_required = 2 * GIB\n    if same_volume:\n        passed = min(cache_free, output_free) >= cache_required + output_required\n    else:\n        passed = cache_free >= cache_required and output_free >= output_required\n    return dict(passed=passed, missing_weight_bytes=missing_bytes,\n                cache_required_bytes=cache_required, output_required_bytes=output_required,\n                cache_free_bytes=cache_free, output_free_bytes=output_free,\n                shared_volume=same_volume,\n                note=\'Reserve 1 GiB for download overhead when needed and 2 GiB for run/checkpoint/ZIP; no base copy in the ZIP.\')\n\n\ndef check_training_storage(config, cache_dir, output, *, allow_download=False):\n    from huggingface_hub import HfApi, hf_hub_download, try_to_load_from_cache\n    cache = Path(cache_dir).expanduser().resolve()\n    output = Path(output).resolve()\n    cache.mkdir(parents=True, exist_ok=True)\n    identity = dict(repo_id=config[\'model_id\'], revision=config[\'revision\'], cache_dir=str(cache))\n    if allow_download:\n        # Metadata only: no safetensors are downloaded until the budget passes.\n        info = HfApi().model_info(config[\'model_id\'], revision=config[\'revision\'],\n                                  files_metadata=True, timeout=30)\n        require(info.sha == config[\'revision\'], \'WEIGHT_METADATA_REVISION\')\n        weights = {item.rfilename: item.size for item in info.siblings\n                   if re.fullmatch(r\'model(?:-\\d+-of-\\d+)?\\.safetensors\', item.rfilename)}\n        require(0 < len(weights) <= 16 and all(type(n) is int and n > 0 for n in weights.values()),\n                \'SAFETENSORS_SIZE_METADATA_REQUIRED\')\n    else:\n        index = hf_hub_download(filename=\'model.safetensors.index.json\',\n                                local_files_only=True, **identity)\n        names = set(read_json(index)[\'weight_map\'].values())\n        weights = {}\n        for name in names:\n            require(re.fullmatch(r\'model-\\d+-of-\\d+\\.safetensors\', name) is not None,\n                    \'WEIGHT_INDEX_FILENAME\')\n            cached = try_to_load_from_cache(filename=name, **identity)\n            require(isinstance(cached, str) and Path(cached).is_file(), \'BASE_NOT_FULLY_CACHED\')\n            weights[name] = Path(cached).stat().st_size\n    cached_bytes = 0\n    for name, size in weights.items():\n        cached = try_to_load_from_cache(filename=name, **identity)\n        if isinstance(cached, str) and Path(cached).is_file() and Path(cached).stat().st_size == size:\n            cached_bytes += size\n    total = sum(weights.values())\n    result = budget(total - cached_bytes, shutil.disk_usage(cache).free,\n                    shutil.disk_usage(output).free, cache.stat().st_dev == output.stat().st_dev)\n    return dict(result, cache_dir=str(cache), model_id=config[\'model_id\'], revision=config[\'revision\'],\n                total_weight_bytes=total, cached_weight_bytes=cached_bytes, weight_files=weights,\n                weights_downloaded_by_check=False, offline=not allow_download)\n', 'smoke.py': '"""Audit and package a completed two-step GPU smoke, without loading Qwen again."""\nimport argparse\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom .data import digest, read_json, read_rows, require, sha_file\nfrom .runtime import inventory, verify_inventory, write_json\n\n\ndef audit(run):\n    return audit_training(run, profile=\'smoke\', steps=2)\n\n\ndef audit_training(run, *, profile, steps):\n    run = Path(run).resolve()\n    manifest = read_json(run / \'run-manifest.json\')\n    require(manifest[\'status\'] == \'TRAINING_COMPLETED\' and manifest[\'profile\'] == profile,\n            \'NOT_COMPLETED_GPU_SMOKE\')\n    verify_inventory(run, manifest[\'files\'])\n    require(set(inventory(run, (\'run-manifest.json\', \'SHA256SUMS.txt\'))) == set(manifest[\'files\']),\n            \'SMOKE_UNLISTED_FILES\')\n    sums = {}\n    for line in (run / \'SHA256SUMS.txt\').read_text(encoding=\'utf-8\').splitlines():\n        sha, name = line.split(\'  \', 1)\n        require(name not in sums, \'SMOKE_DUPLICATE_CHECKSUM\')\n        sums[name] = sha\n    require(set(sums) == set(manifest[\'files\']) | {\'run-manifest.json\'}\n            and all(sha_file(run / name) == sha for name, sha in sums.items()), \'SMOKE_CHECKSUM_LIST\')\n    config = read_json(run / \'resolved-config.json\')\n    require(digest(config) == manifest[\'config_sha256\'], \'SMOKE_CONFIG_HASH\')\n    reference = read_json(run / \'data-reference.json\')\n    require(reference[\'manifest_sha256\'] == config[\'data_manifest_sha256\']\n            == manifest[\'data_manifest_sha256\']\n            and reference[\'contract_sha256\'] == config[\'contract_sha256\']\n            and (reference[\'train\'], reference[\'validation\'], reference[\'holdout\']) == (128, 32, 0),\n            \'SMOKE_DATA_IDENTITY\')\n    args = read_json(run / \'training-arguments.json\')\n    require(args[\'max_steps\'] == steps and args[\'fp16\'] is True and args[\'bf16\'] is False\n            and args[\'gradient_accumulation_steps\'] == 4\n            and args[\'per_device_train_batch_size\'] == 1, \'SMOKE_TRAINING_ARGUMENTS\')\n    env = read_json(run / \'environment.json\')\n    require(env[\'cuda_available\'] is True and env[\'cuda\'] is not None\n            and len(env[\'gpus\']) == 1 and \'T4\' in env[\'gpus\'][0], \'SMOKE_GPU_EVIDENCE_REQUIRED\')\n    devices = read_json(run / \'training-device.json\')\n    require(devices[\'visible_gpu_count\'] == devices[\'world_size\'] == 1\n            and devices[\'mode\'] == \'SINGLE_GPU_NO_DATAPARALLEL\', \'SMOKE_SINGLE_GPU_REQUIRED\')\n    loaded = read_json(run / \'model-load-report.json\')\n    require(loaded[\'model_id\'] == config[\'model_id\'] and loaded[\'revision\'] == config[\'revision\']\n            and loaded[\'is_loaded_in_4bit\'] is True and loaded[\'use_cache\'] is False\n            and loaded[\'quantization\'][\'bnb_4bit_quant_type\'] == \'nf4\'\n            and loaded[\'quantization\'][\'bnb_4bit_use_double_quant\'] is True, \'SMOKE_BASE_LOAD_IDENTITY\')\n    update = read_json(run / \'adapter-update-report.json\')\n    require((update[\'start_global_step\'], update[\'global_step\'], update[\'attempted_updates\'],\n             update[\'applied_updates\'], update[\'scaler_skipped_updates\'], update[\'finite_gradient_checks\'])\n            == (0, steps, steps, steps, 0, steps) and math.isfinite(update[\'adapter_delta_l2\'])\n            and update[\'adapter_delta_l2\'] > 0 and update[\'base_frozen\'] is True,\n            \'SMOKE_OPTIMIZER_PROOF\')\n    events = read_rows(run / \'optimizer-events.jsonl\')\n    require([r[\'step\'] for r in events] == list(range(1, steps + 1))\n            and all(r[\'update_applied\'] is True for r in events), \'SMOKE_UPDATE_EVENTS\')\n    logs = read_rows(run / \'train-metrics.jsonl\')\n    loss = {r[\'step\']: r[\'loss\'] for r in logs if \'loss\' in r}\n    require(set(loss) == set(range(1, steps + 1)) and all(math.isfinite(v) for v in loss.values()), \'SMOKE_FINITE_LOSS\')\n    evaluation = read_json(run / \'eval-metrics.json\')\n    require(math.isfinite(evaluation[\'eval_loss\']), \'SMOKE_FINITE_EVAL_LOSS\')\n    require(read_json(run / \'storage-check.json\')[\'passed\'] is True, \'SMOKE_STORAGE_CHECK\')\n    parameters = read_json(run / \'trainable-parameters.json\')\n    require(parameters[\'base_frozen\'] is True and parameters[\'trainable_parameters\'] > 0,\n            \'SMOKE_BASE_FREEZE\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\', \'binding.json\'}\n    checkpoint = run / (\'checkpoint-\' + str(steps))\n    require(checkpoint.is_dir() and required <= {p.name for p in checkpoint.iterdir()},\n            \'SMOKE_CHECKPOINT_INCOMPLETE\')\n    require(read_json(checkpoint / \'trainer_state.json\')[\'global_step\'] == steps,\n            \'SMOKE_CHECKPOINT_STEP\')\n    binding = read_json(checkpoint / \'binding.json\')\n    require(binding[\'config_sha256\'] == digest(config)\n            and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\'], \'SMOKE_CHECKPOINT_BINDING\')\n    from safetensors import safe_open\n    import torch\n    with safe_open(run / \'final-adapter/adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as final, \\\n         safe_open(checkpoint / \'adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as saved:\n        require(set(final.keys()) == set(saved.keys()) and bool(final.keys()), \'SMOKE_SAVED_TENSOR_KEYS\')\n        count = 0\n        for name in final.keys():\n            require(\'lora_\' in name, \'SMOKE_UNEXPECTED_ADAPTER_TENSOR\')\n            tensor = final.get_tensor(name)\n            require(torch.isfinite(tensor).all().item() and torch.equal(tensor, saved.get_tensor(name)),\n                    \'SMOKE_SAVED_TENSORS_DIFFER\')\n            count += tensor.numel()\n    require(count == parameters[\'trainable_parameters\'], \'SMOKE_PARAMETER_COUNT\')\n    vram = read_json(run / \'vram.json\')\n    require(0 < vram[\'peak_allocated_bytes\'] <= vram[\'peak_reserved_bytes\'], \'SMOKE_VRAM_METRICS\')\n    return dict(status=\'GPU_SMOKE_PASSED\' if profile == \'smoke\' else \'GPU_PILOT_PASSED\',\n                global_step=steps, applied_updates=steps,\n                scaler_skipped_updates=0, adapter_delta_l2=update[\'adapter_delta_l2\'], base_frozen=True,\n                train_loss_by_step=loss, eval_loss=evaluation[\'eval_loss\'],\n                trainable_parameters=count, checkpoint=checkpoint.name, saved_adapter_tensors_match=True,\n                vram=vram, config_sha256=digest(config), data_manifest_sha256=config[\'data_manifest_sha256\'],\n                run_manifest_sha256=sha_file(run / \'run-manifest.json\'),\n                limitation=(\'Two-step technical smoke, not pilot 50, reload/resume verification or quality improvement.\'\n                            if profile == \'smoke\' else\n                            \'Technical pilot 50, not independent GPU reload/resume verification or quality improvement.\'))\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    run = args.run.resolve()\n    report = audit(run)\n    report_path = run.with_name(run.name + \'-audit.json\')\n    if report_path.exists():\n        require(read_json(report_path) == report, \'SMOKE_AUDIT_OUTPUT_DIFFERS\')\n    else:\n        write_json(report_path, report)\n    archive = run.with_name(run.name + \'.zip\')\n    require(not archive.exists(), \'SMOKE_ZIP_EXISTS\')\n    needed = sum(p.stat().st_size for p in run.rglob(\'*\') if p.is_file())\n    require(shutil.disk_usage(run.parent).free >= needed + 256 * 1024 ** 2, \'INSUFFICIENT_SPACE_FOR_SMOKE_ZIP\')\n    with ZipFile(archive, \'x\', compression=ZIP_DEFLATED) as zipped:\n        for path in sorted(run.rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, path.relative_to(run).as_posix())\n    write_json(run.with_name(run.name + \'-receipt.json\'), dict(\n        file=archive.name, sha256=sha_file(archive), size_bytes=archive.stat().st_size,\n        audit_file=report_path.name, audit_sha256=sha_file(report_path),\n        run_manifest_sha256=report[\'run_manifest_sha256\'], model_revision=read_json(run / \'resolved-config.json\')[\'revision\'],\n        global_step=2, scope=\'PRIVATE_GPU_SMOKE_HANDOFF\', base_weights_included=False))\n    print(\'GPU_SMOKE_PASSED: \' + str(report_path))\n    print(\'Private artifact: \' + str(archive))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'pilot.py': '"""Audit and export the fresh 50-step pilot, without loading Qwen or pickle state."""\nimport argparse\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom .data import digest, read_json, read_rows, require, sha_file\nfrom .runtime import validate_recipe, write_json\nfrom .smoke import audit_training\n\n\ndef audit(run):\n    run = Path(run).resolve()\n    report = audit_training(run, profile=\'pilot\', steps=50)\n    config = read_json(run / \'resolved-config.json\')\n    validate_recipe(config, \'pilot\')\n    recipe = Path(__file__).resolve().parents[2] / \'configs/qlora-pilot.json\'\n    require(digest(config) == digest(read_json(recipe)), \'PILOT_RECIPE_IDENTITY\')\n    reference = read_json(run / \'data-reference.json\')\n    require(reference.get(\'resume_checkpoint\') is None\n            and reference.get(\'resume_manifest_sha256\') is None, \'PILOT_MUST_START_FRESH\')\n    args = read_json(run / \'training-arguments.json\')\n    require(args[\'save_steps\'] == args[\'eval_steps\'] == 25\n            and args[\'save_total_limit\'] == 2, \'PILOT_CHECKPOINT_SCHEDULE\')\n    require(all(args.get(key) == value for key, value in config[\'training\'].items())\n            and args.get(\'seed\') == args.get(\'data_seed\') == config[\'seed\']\n            and args.get(\'gradient_checkpointing\') is True, \'PILOT_TRAINING_RECIPE\')\n    evaluations = {r[\'step\']: r[\'eval_loss\'] for r in read_rows(run / \'train-metrics.jsonl\')\n                   if \'eval_loss\' in r}\n    require(set(evaluations) == {25, 50}\n            and all(math.isfinite(v) for v in evaluations.values()), \'PILOT_EVALUATION_STEPS\')\n    require(evaluations[50] == report[\'eval_loss\'], \'PILOT_FINAL_EVAL_MISMATCH\')\n    expected = {\'checkpoint-25\', \'checkpoint-50\'}\n    require({p.name for p in run.glob(\'checkpoint-*\')} == expected, \'PILOT_CHECKPOINT_SET\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\', \'binding.json\'}\n    from safetensors import safe_open\n    import torch\n    for step in (25, 50):\n        folder = run / (\'checkpoint-\' + str(step))\n        require(required <= {p.name for p in folder.iterdir()}, \'PILOT_CHECKPOINT_INCOMPLETE\')\n        require(read_json(folder / \'trainer_state.json\')[\'global_step\'] == step, \'PILOT_CHECKPOINT_STEP\')\n        binding = read_json(folder / \'binding.json\')\n        require(binding[\'config_sha256\'] == digest(config)\n                and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\']\n                and binding[\'global_step\'] == step, \'PILOT_CHECKPOINT_BINDING\')\n        with safe_open(folder / \'adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as tensors:\n            require(bool(tensors.keys()) and all(\'lora_\' in key\n                    and torch.isfinite(tensors.get_tensor(key)).all().item() for key in tensors.keys()),\n                    \'PILOT_CHECKPOINT_TENSORS\')\n        adapter = read_json(folder / \'adapter_config.json\')\n        require(adapter[\'base_model_name_or_path\'] == config[\'model_id\']\n                and adapter[\'revision\'] == config[\'revision\']\n                and all(adapter[key] == config[\'lora\'][key]\n                        for key in (\'r\', \'lora_alpha\', \'lora_dropout\', \'bias\', \'task_type\')),\n                \'PILOT_ADAPTER_IDENTITY\')\n    report.update(checkpoints=sorted(expected), eval_loss_by_step=evaluations,\n                  initialization=\'PINNED_BASE_WITH_FRESH_LORA\', gpu_reload=\'NOT_RUN\',\n                  gpu_resume=\'NOT_RUN\', quality_improvement=\'NOT_ESTABLISHED\')\n    return report\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    run = args.run.resolve()\n    report = audit(run)\n    report_path = run.with_name(run.name + \'-audit.json\')\n    if report_path.exists():\n        require(read_json(report_path) == read_json_value(report), \'PILOT_AUDIT_OUTPUT_DIFFERS\')\n    else:\n        write_json(report_path, report)\n    archive = run.with_name(run.name + \'.zip\')\n    receipt = run.with_name(run.name + \'-receipt.json\')\n    require(not archive.exists() and not receipt.exists(), \'PILOT_EXPORT_EXISTS\')\n    needed = sum(p.stat().st_size for p in run.rglob(\'*\') if p.is_file())\n    require(shutil.disk_usage(run.parent).free >= needed + 256 * 1024 ** 2,\n            \'INSUFFICIENT_SPACE_FOR_PILOT_ZIP\')\n    with ZipFile(archive, \'x\', compression=ZIP_DEFLATED) as zipped:\n        for path in sorted(run.rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, path.relative_to(run).as_posix())\n    write_json(receipt, dict(file=archive.name, sha256=sha_file(archive),\n        size_bytes=archive.stat().st_size, audit_file=report_path.name,\n        audit_sha256=sha_file(report_path), run_manifest_sha256=report[\'run_manifest_sha256\'],\n        model_revision=read_json(run / \'resolved-config.json\')[\'revision\'], global_step=50,\n        scope=\'PRIVATE_GPU_PILOT_HANDOFF\', base_weights_included=False))\n    print(\'GPU_PILOT_PASSED: \' + str(report_path))\n    print(\'Private artifact: \' + str(archive))\n    return 0\n\n\ndef read_json_value(value):\n    # Normalize integer step keys to the representation stored in JSON reports.\n    import json\n    return json.loads(json.dumps(value, allow_nan=False))\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'assessment.py': '"""Private development comparison and authenticated pilot reload/resume evidence."""\nimport argparse\nfrom collections import Counter\nfrom contextlib import nullcontext\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport sys\nimport time\nimport unicodedata\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom .data import (authenticate, digest, load_pilot, loads, read_json, read_rows,\n                   require, sha_file, within)\nfrom .pilot import audit as audit_pilot\nfrom .runtime import environment, inventory, load_training_model, verify_inventory, write_json\n\n\nPILOT_NAME = \'pilot-20261104T180615021055Z\'\nPILOT_SHA = \'b228645e1df2d4370715539399f388208d1c7aa04745ce85ca7ccb94425a2a1d\'\nZIP_SHA = \'d7ae6f75c8620bc1463b91ca32fe9fca98b89e261c7e4cc5611736560acd215f\'\nGENERATION = dict(max_new_tokens=512, do_sample=False, num_beams=1, use_cache=True,\n                  eos_token_id=151645, pad_token_id=151643, max_time=90.0)\n\n\ndef check_run(folder, manifest_sha):\n    folder = Path(folder).resolve()\n    require(sha_file(folder / \'run-manifest.json\') == manifest_sha, \'RUN_MANIFEST_IDENTITY\')\n    manifest = read_json(folder / \'run-manifest.json\')\n    verify_inventory(folder, manifest[\'files\'])\n    require(set(inventory(folder, (\'run-manifest.json\', \'SHA256SUMS.txt\'))) == set(manifest[\'files\']),\n            \'UNLISTED_RUN_FILES\')\n    return manifest\n\n\ndef find_pilot(work, search):\n    direct = Path(work) / \'runs\' / PILOT_NAME\n    if (direct / \'run-manifest.json\').is_file():\n        check_run(direct, PILOT_SHA)\n        return direct.resolve()\n    candidates = [p.parent for p in Path(search).rglob(\'run-manifest.json\')\n                  if sha_file(p) == PILOT_SHA]\n    if candidates:\n        require(len(candidates) == 1, \'MULTIPLE_PILOT_COPIES_SELECT_EXPLICITLY\')\n        check_run(candidates[0], PILOT_SHA)\n        return candidates[0].resolve()\n    archives = [p for p in Path(search).rglob(PILOT_NAME + \'.zip\') if sha_file(p) == ZIP_SHA]\n    require(len(archives) == 1, \'PILOT_MISSING_ADD_ORIGINAL_PRIVATE_ZIP_INPUT\')\n    require(not direct.exists(), \'PILOT_EXTRACTION_DESTINATION_EXISTS\')\n    require(shutil.disk_usage(Path(work)).free > 3 * 1024 ** 3, \'PILOT_EXTRACTION_DISK\')\n    with ZipFile(archives[0]) as zipped:\n        entries = zipped.infolist()\n        names = [e.filename for e in entries]\n        require(len(entries) <= 250 and len(names) == len(set(n.casefold() for n in names))\n                and sum(e.file_size for e in entries) < 2 * 1024 ** 3, \'ZIP_LIMITS\')\n        for entry in entries:\n            require(not entry.is_dir() and not stat.S_ISLNK(entry.external_attr >> 16)\n                    and PurePosixPath(entry.filename).as_posix() == entry.filename\n                    and \'..\' not in PurePosixPath(entry.filename).parts, \'ZIP_MEMBER\')\n            within(direct, entry.filename)\n        require(zipped.testzip() is None, \'ZIP_CRC\')\n        require(hashlib.sha256(zipped.read(\'run-manifest.json\')).hexdigest() == PILOT_SHA, \'ZIP_MANIFEST\')\n        direct.mkdir(parents=True)\n        for entry in entries:\n            target = within(direct, entry.filename)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with zipped.open(entry) as stream, target.open(\'xb\') as output:\n                shutil.copyfileobj(stream, output)\n    check_run(direct, PILOT_SHA)\n    return direct.resolve()\n\n\ndef cases_from_bundle(bundle, config):\n    folder = Path(bundle) / \'data/pilot-v2\'\n    authenticate(folder, config)\n    selection = read_json(folder / \'evaluation-selection.json\')\n    ids = selection[\'record_ids\']\n    require(selection[\'scope\'] == \'DEVELOPMENT_VALIDATION_NOT_HELDOUT_TEST\'\n            and selection[\'selected_before_adapter_outputs\'] is True\n            and selection[\'count\'] == len(ids) == len(set(ids)) == 20, \'LOCKED_EVALUATION_SELECTION\')\n    payloads = read_rows(folder / \'validation.jsonl\')\n    metas = read_rows(folder / \'validation.metadata.jsonl\')\n    lookup = {}\n    for payload, meta in zip(payloads, metas):\n        require(digest(payload) == meta[\'training_payload_sha256\'], \'EVAL_PAYLOAD_HASH\')\n        lookup[meta[\'source_record_id\']] = (payload, meta)\n    require(set(ids) <= set(lookup), \'EVAL_NOT_VALIDATION\')\n    cases = []\n    for rid in ids:\n        payload, meta = lookup[rid]\n        require(meta[\'split\'] == \'validation\' and len(payload[\'completion\']) == 1\n                and payload[\'completion\'][0][\'role\'] == \'assistant\', \'EVAL_PAYLOAD\')\n        public = loads(payload[\'prompt\'][-1][\'content\'])\n        cases.append(dict(record_id=rid, task_type=meta[\'task_type\'], prompt=payload[\'prompt\'],\n            reference=payload[\'completion\'][0][\'content\'], evidence=public[\'evidence\'],\n            payload_sha256=meta[\'training_payload_sha256\']))\n    return cases\n\n\ndef automatic_checks(text, case):\n    result = dict(format_valid=False, citation_ids_valid=None, expected_behavior_match=False,\n                  reference_exact_match=False, reference_token_f1=0.0,\n                  semantic_correctness=None, grounded=None, citation_supports_claim=None)\n    if case[\'task_type\'] == \'hint\':\n        match = re.fullmatch(r\'(READ|OUTLINE) \\[([1-9]\\d*)\\]|COMPARE \\[([1-9]\\d*)\\] \\[([1-9]\\d*)\\]\', text.strip())\n        visible = {e[\'citation_label\']: e for e in case[\'evidence\']}\n        if match:\n            labels = [int(match[2])] if match[2] else [int(match[3]), int(match[4])]\n            valid = all(label in visible for label in labels)\n            if len(labels) == 2 and valid:\n                valid = labels[0] != labels[1] and visible[labels[0]].get(\'source_id\', labels[0]) != visible[labels[1]].get(\'source_id\', labels[1])\n            result.update(format_valid=True, citation_ids_valid=valid, expected_behavior_match=valid)\n        result[\'reference_exact_match\'] = text.strip() == case[\'reference\'].strip()\n        if text.strip() == \'INSUFFICIENT_EVIDENCE\':\n            result.update(format_valid=True, expected_behavior_match=result[\'reference_exact_match\'])\n        # Action-only grammar checks absence of extra answer text, not usefulness of the hint.\n        result[\'hint_action_only\'] = bool(match)\n    else:\n        try:\n            value = loads(text.strip())\n            require(isinstance(value, dict) and set(value) == {\'items\'}\n                    and isinstance(value[\'items\'], list) and len(value[\'items\']) <= 24, \'OUTPUT_SCHEMA\')\n            for item in value[\'items\']:\n                require(isinstance(item, dict) and set(item) == {\'text\', \'evidence_id\'}\n                        and isinstance(item[\'text\'], str) and 0 < len(item[\'text\'].strip()) <= 1500\n                        and \'\\n\' not in item[\'text\'] and isinstance(item[\'evidence_id\'], str), \'OUTPUT_ITEM\')\n            ids = {e[\'evidence_id\'] for e in case[\'evidence\']}\n            valid = all(item[\'evidence_id\'] in ids for item in value[\'items\'])\n            result.update(format_valid=True, citation_ids_valid=valid if value[\'items\'] else None,\n                expected_behavior_match=valid and (bool(value[\'items\']) != (case[\'task_type\'] == \'insufficient_evidence\')),\n                reference_exact_match=value == loads(case[\'reference\']))\n        except (ValueError, TypeError, KeyError):\n            pass\n    def tokens(value):\n        return Counter(re.findall(r\'\\w+\', unicodedata.normalize(\'NFC\', value).casefold()))\n    predicted, target = tokens(text), tokens(case[\'reference\'])\n    overlap = sum((predicted & target).values())\n    denominator = sum(predicted.values()) + sum(target.values())\n    result[\'reference_token_f1\'] = 2 * overlap / denominator if denominator else 0.0\n    return result\n\n\ndef prompt_ids(case, tokenizer, contract):\n    ids = tokenizer.apply_chat_template(case[\'prompt\'], tokenize=True, add_generation_prompt=True)\n    require(len(ids) <= contract[\'token_limits\'][\'max_prompt_tokens\'], \'GENERATION_PROMPT_BUDGET\')\n    return ids\n\n\ndef score_generation(text, case, status):\n    require(status in (\'COMPLETED\', \'OUTPUT_LIMIT\', \'INCOMPLETE_OR_TIMEOUT\'), \'GENERATION_STATUS\')\n    checks = automatic_checks(text, case)\n    if status != \'COMPLETED\':\n        checks.update(format_valid=False, citation_ids_valid=False, expected_behavior_match=False,\n                      reference_exact_match=False, reference_token_f1=0.0)\n    return checks\n\n\ndef verify_loaded_adapter(model, adapter):\n    import torch\n    from peft import get_peft_model_state_dict\n    from safetensors.torch import load_file\n    stored = load_file(Path(adapter) / \'adapter_model.safetensors\', device=\'cpu\')\n    restored = get_peft_model_state_dict(model, adapter_name=\'pilot\')\n    require(set(stored) == set(restored) and all(torch.equal(stored[k], restored[k].detach().cpu()) for k in stored),\n            \'GPU_RELOADED_ADAPTER_TENSORS_DIFFER\')\n\n\ndef summarize(rows):\n    require(len(rows) == 40, \'FORTY_PREDICTIONS_REQUIRED\')\n    result = {}\n    for arm in (\'base\', \'adapter\'):\n        subset = [r for r in rows if r[\'arm\'] == arm]\n        require(len(subset) == 20 and len({r[\'record_id\'] for r in subset}) == 20, \'ARM_COVERAGE\')\n        citations = [r for r in subset if r[\'checks\'][\'citation_ids_valid\'] is not None]\n        result[arm] = dict(count=20, completed=sum(r[\'status\'] == \'COMPLETED\' for r in subset),\n            format_valid=sum(r[\'checks\'][\'format_valid\'] for r in subset),\n            expected_behavior_match=sum(r[\'checks\'][\'expected_behavior_match\'] for r in subset),\n            citation_ids_valid=sum(r[\'checks\'][\'citation_ids_valid\'] is True for r in citations),\n            citation_eligible_count=len(citations),\n            reference_exact_match=sum(r[\'checks\'][\'reference_exact_match\'] for r in subset),\n            mean_reference_token_f1=sum(r[\'checks\'][\'reference_token_f1\'] for r in subset) / 20,\n            mean_latency_seconds=sum(r[\'latency_seconds\'] for r in subset) / 20,\n            input_tokens=sum(r[\'input_tokens\'] for r in subset),\n            output_tokens=sum(r[\'output_tokens\'] for r in subset))\n    return dict(arms=result, scope=\'20_DEVELOPMENT_VALIDATION_CASES_NOT_HELDOUT_TEST\',\n        semantic_quality=\'NOT_REVIEWED\', human_review_complete=False,\n        limitation=\'Format, valid IDs, action grammar and lexical overlap do not establish factuality, grounding or citation support. Failed/truncated generations count as failures.\')\n\n\ndef prepare(args):\n    require(not args.output.exists(), \'ASSESSMENT_OUTPUT_EXISTS\')\n    pilot = find_pilot(args.work, args.search)\n    report = audit_pilot(pilot)\n    config = read_json(pilot / \'resolved-config.json\')\n    cases = cases_from_bundle(args.bundle, config)\n    manifest = read_json(pilot / \'run-manifest.json\')\n    frozen = args.frozen_tools / \'scripts/qlora\'\n    actual = {p.name: sha_file(p) for p in frozen.glob(\'*.py\')}\n    require(actual == manifest[\'source_files_sha256\'], \'FROZEN_TRAINER_SOURCE_MISMATCH\')\n    for module, expected in actual.items():\n        require(sha_file(pilot / \'source/qlora\' / module) == expected, \'PILOT_SOURCE_HASH\')\n    for path in (pilot, args.bundle, args.frozen_tools):\n        require(not args.output.resolve().is_relative_to(path.resolve())\n                and not path.resolve().is_relative_to(args.output.resolve()), \'OUTPUT_OVERLAPS_INPUT\')\n    args.output.mkdir(parents=True)\n    source = args.output / \'source/assessment\'\n    source.mkdir(parents=True)\n    for module in Path(__file__).parent.glob(\'*.py\'):\n        (source / module.name).write_bytes(module.read_bytes())\n    write_json(args.output / \'context.json\', dict(pilot=str(pilot), pilot_manifest_sha256=PILOT_SHA,\n        bundle=str(args.bundle.resolve()), frozen_tools=str(args.frozen_tools.resolve()),\n        config_sha256=digest(config), evaluation_ids=[c[\'record_id\'] for c in cases],\n        generation=GENERATION, initialization=\'RELOAD_FINAL_ADAPTER_NOT_NEW_TRAINING\',\n        scope=\'PRIVATE_DEVELOPMENT_CHECK\', pilot_audit=report,\n        source_files_sha256={p.name: sha_file(p) for p in Path(__file__).parent.glob(\'*.py\')}))\n    print(\'ASSESSMENT_INPUT_READY: \' + str(args.output))\n\n\ndef load_context(output):\n    context = read_json(output / \'context.json\')\n    check_run(context[\'pilot\'], PILOT_SHA)\n    config = read_json(Path(context[\'pilot\']) / \'resolved-config.json\')\n    require(context[\'pilot_manifest_sha256\'] == PILOT_SHA and context[\'config_sha256\'] == digest(config),\n            \'ASSESSMENT_CONTEXT_IDENTITY\')\n    require(context[\'generation\'] == GENERATION, \'GENERATION_CONFIG_CHANGED\')\n    actual = {p.name: sha_file(p) for p in Path(__file__).parent.glob(\'*.py\')}\n    require(actual == context[\'source_files_sha256\'], \'ASSESSMENT_CODE_CHANGED\')\n    for name, expected in actual.items():\n        require(sha_file(output / \'source/assessment\' / name) == expected, \'ASSESSMENT_SOURCE_SNAPSHOT\')\n    return context, config\n\n\ndef evaluate(args):\n    context, config = load_context(args.output)\n    target = args.output / \'evaluation\'\n    require(not target.exists(), \'EVALUATION_ALREADY_EXISTS\')\n    target.mkdir()\n    import torch\n    from transformers import GenerationConfig, set_seed\n    torch.set_num_threads(2)\n    data, tokenizer, contract = load_pilot(Path(context[\'bundle\']) / \'data/pilot-v2\',\n                                          Path(context[\'bundle\']) / \'tokenizer\', config)\n    cases = cases_from_bundle(context[\'bundle\'], config)\n    write_json(target / \'cases.json\', cases)\n    actual_environment = environment()\n    require(actual_environment[\'torch\'] == read_json(Path(context[\'pilot\']) / \'environment.json\')[\'torch\'],\n            \'TORCH_VERSION_DIFFERS_FROM_PILOT\')\n    write_json(target / \'environment.json\', actual_environment)\n    write_json(target / \'generation-config.json\', GENERATION)\n    set_seed(config[\'seed\'])\n    model = load_training_model(config, cache_dir=args.cache_dir, allow_download=args.allow_model_download)\n    adapter = Path(context[\'pilot\']) / \'final-adapter\'\n    model.load_adapter(str(adapter), adapter_name=\'pilot\', is_trainable=False)\n    model.set_adapter(\'pilot\')\n    model.requires_grad_(False)\n    model.gradient_checkpointing_disable()\n    model.config.use_cache = True\n    model.eval()\n    verify_loaded_adapter(model, adapter)\n    example = {k: torch.tensor([v], dtype=torch.long, device=\'cuda:0\') for k, v in data[\'validation\'][0].items()}\n    with torch.inference_mode():\n        reload_loss = model(**example).loss.item()\n    require(math.isfinite(reload_loss), \'GPU_RELOAD_FORWARD_NONFINITE\')\n    write_json(target / \'reload.json\', dict(status=\'GPU_RELOAD_PASSED\',\n        process_id=os.getpid(), reloaded_saved_tensors_exact=True, forward_loss=reload_loss,\n        model_id=config[\'model_id\'], revision=config[\'revision\'], quantization=\'NF4_DOUBLE_QUANT_FP16\',\n        adapter_sha256=sha_file(adapter / \'adapter_model.safetensors\'),\n        base_comparison=\'SAME_NF4_BASE_WITH_ALL_LORA_DISABLED\', requires_grad_any=any(p.requires_grad for p in model.parameters())))\n    del example\n    generation = GenerationConfig(**{k: v for k, v in GENERATION.items() if k != \'max_time\'})\n\n    def generate(case, arm):\n        # Only prompt messages reach generate; reference answers stay outside model inputs.\n        ids = prompt_ids(case, tokenizer, contract)\n        inputs = torch.tensor([ids], dtype=torch.long, device=\'cuda:0\')\n        set_seed(config[\'seed\'])\n        enabled = model.disable_adapter() if arm == \'base\' else nullcontext()\n        with enabled, torch.inference_mode():\n            torch.cuda.synchronize()\n            started = time.monotonic()\n            generated = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),\n                generation_config=generation, max_time=GENERATION[\'max_time\'])\n            torch.cuda.synchronize()\n            elapsed = time.monotonic() - started\n        continuation = generated[0, len(ids):].detach().cpu().tolist()\n        raw = tokenizer.decode(continuation, skip_special_tokens=True)\n        ended = bool(continuation) and continuation[-1] == tokenizer.eos_token_id\n        status = \'COMPLETED\' if ended else (\'OUTPUT_LIMIT\' if len(continuation) >= GENERATION[\'max_new_tokens\'] else \'INCOMPLETE_OR_TIMEOUT\')\n        checks = score_generation(raw, case, status)\n        return dict(record_id=case[\'record_id\'], task_type=case[\'task_type\'], arm=arm,\n            prompt_sha256=digest(case[\'prompt\']), payload_sha256=case[\'payload_sha256\'],\n            status=status, raw_output=raw, input_tokens=len(ids), output_tokens=len(continuation),\n            latency_seconds=elapsed, checks=checks)\n\n    for arm in (\'base\', \'adapter\'):\n        generate(cases[0], arm)\n    write_json(target / \'warmup.json\', dict(calls=2, case_id=cases[0][\'record_id\'], excluded_from_timing_metrics=True))\n    rows = []\n    with (target / \'predictions.jsonl\').open(\'x\', encoding=\'utf-8\') as stream:\n        for index, case in enumerate(cases):\n            for arm in ((\'base\', \'adapter\') if index % 2 == 0 else (\'adapter\', \'base\')):\n                row = generate(case, arm)\n                stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + \'\\n\')\n                stream.flush()\n                rows.append(row)\n                write_json(args.output / \'phase.json\', dict(phase=\'COMPARING\', completed=len(rows), total=40,\n                           record_id=case[\'record_id\'], arm=arm))\n    write_json(target / \'comparison.json\', summarize(rows))\n    fields = [\'record_id\', \'arm\', \'content_correct\', \'grounded\', \'citation_supports_claim\',\n              \'hint_no_answer_leak\', \'decision\', \'notes\']\n    with (target / \'review.csv\').open(\'x\', encoding=\'utf-8\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        for row in rows:\n            writer.writerow(dict(record_id=row[\'record_id\'], arm=row[\'arm\']))\n    write_json(target / \'vram.json\', dict(peak_allocated_bytes=torch.cuda.max_memory_allocated(0),\n                                         peak_reserved_bytes=torch.cuda.max_memory_reserved(0)))\n    print(\'GPU_RELOAD_AND_DEVELOPMENT_COMPARISON_COMPLETED\')\n\n\ndef audit_resume(output):\n    context, config = load_context(output)\n    run = output / \'resume\'\n    manifest = read_json(run / \'run-manifest.json\')\n    check_run(run, sha_file(run / \'run-manifest.json\'))\n    require(manifest[\'status\'] == \'TRAINING_COMPLETED\' and manifest[\'profile\'] == \'pilot\'\n            and manifest[\'config_sha256\'] == digest(config), \'RESUME_NOT_COMPLETED\')\n    original = Path(context[\'pilot\'])\n    require(manifest[\'source_files_sha256\'] == read_json(original / \'run-manifest.json\')[\'source_files_sha256\'],\n            \'RESUME_SOURCE_CHANGED\')\n    args = read_json(run / \'training-arguments.json\')\n    require(args[\'max_steps\'] == 50 and args[\'fp16\'] is True and args[\'bf16\'] is False\n            and all(args[key] == value for key, value in config[\'training\'].items()),\n            \'RESUME_SCHEDULE_MUST_REMAIN_50\')\n    env = read_json(run / \'environment.json\')\n    devices = read_json(run / \'training-device.json\')\n    require(env[\'cuda_available\'] is True and len(env[\'gpus\']) == 1 and \'T4\' in env[\'gpus\'][0]\n            and devices[\'world_size\'] == devices[\'visible_gpu_count\'] == 1, \'RESUME_GPU_REQUIRED\')\n    loaded = read_json(run / \'model-load-report.json\')\n    require(loaded[\'model_id\'] == config[\'model_id\'] and loaded[\'revision\'] == config[\'revision\']\n            and loaded[\'is_loaded_in_4bit\'] is True, \'RESUME_MODEL_IDENTITY\')\n    ref = read_json(run / \'data-reference.json\')\n    require(Path(ref[\'resume_checkpoint\']).resolve() == (original / \'checkpoint-25\').resolve()\n            and ref[\'resume_manifest_sha256\'] == PILOT_SHA, \'RESUME_BINDING\')\n    update = read_json(run / \'adapter-update-report.json\')\n    require((update[\'start_global_step\'], update[\'global_step\'], update[\'applied_updates\'],\n             update[\'attempted_updates\'], update[\'finite_gradient_checks\'], update[\'scaler_skipped_updates\'])\n            == (25, 26, 1, 1, 1, 0) and update[\'base_frozen\'] is True\n            and math.isfinite(update[\'adapter_delta_l2\']) and update[\'adapter_delta_l2\'] > 0, \'RESUME_UPDATE_PROOF\')\n    events = read_rows(run / \'optimizer-events.jsonl\')\n    require(len(events) == 1 and events[0][\'step\'] == 26 and events[0][\'update_applied\'] is True, \'RESUME_EVENTS\')\n    state = read_json(run / \'checkpoint-26/trainer_state.json\')\n    require(state[\'global_step\'] == 26, \'RESUME_CHECKPOINT_26\')\n    import torch\n    # Restricted tensor/primitive loading only. Arbitrary pickle globals are never allowed in audit.\n    before = torch.load(original / \'checkpoint-25/scheduler.pt\', map_location=\'cpu\', weights_only=True)\n    after = torch.load(run / \'checkpoint-26/scheduler.pt\', map_location=\'cpu\', weights_only=True)\n    require(before[\'last_epoch\'] == 25 and after[\'last_epoch\'] == 26\n            and before[\'base_lrs\'] == after[\'base_lrs\'], \'RESUME_SCHEDULER_NOT_RESTORED\')\n    optimizer_before = torch.load(original / \'checkpoint-25/optimizer.pt\', map_location=\'cpu\', weights_only=True)\n    optimizer_after = torch.load(run / \'checkpoint-26/optimizer.pt\', map_location=\'cpu\', weights_only=True)\n    require(bool(optimizer_before[\'state\'])\n            and set(optimizer_before[\'state\']) == set(optimizer_after[\'state\'])\n            and {int(v[\'step\']) for v in optimizer_before[\'state\'].values()} == {25}\n            and {int(v[\'step\']) for v in optimizer_after[\'state\'].values()} == {26},\n            \'RESUME_OPTIMIZER_NOT_RESTORED\')\n    del optimizer_before, optimizer_after\n    require(math.isfinite(read_json(run / \'eval-metrics.json\')[\'eval_loss\']), \'RESUME_EVAL_NONFINITE\')\n    from safetensors.torch import load_file\n    final = load_file(run / \'final-adapter/adapter_model.safetensors\')\n    saved = load_file(run / \'checkpoint-26/adapter_model.safetensors\')\n    require(set(final) == set(saved) and all(torch.isfinite(v).all().item() and torch.equal(v, saved[k])\n            for k, v in final.items()), \'RESUME_SAVED_ADAPTER_MISMATCH\')\n    return dict(status=\'GPU_RESUME_PASSED\', start_global_step=25, global_step=26,\n                applied_updates=1, scheduler_step_before=25, scheduler_step_after=26,\n                optimizer_state_steps_before=[25], optimizer_state_steps_after=[26],\n                original_max_steps=50, adapter_delta_l2=update[\'adapter_delta_l2\'],\n                original_pilot_untouched=True, limitation=\'One resume verification update, not a second pilot.\')\n\n\ndef export(args):\n    context, config = load_context(args.output)\n    comparison = read_json(args.output / \'evaluation/comparison.json\')\n    predictions = read_rows(args.output / \'evaluation/predictions.jsonl\')\n    require(comparison == summarize(predictions), \'COMPARISON_RECOMPUTATION\')\n    cases = cases_from_bundle(context[\'bundle\'], config)\n    require(read_json(args.output / \'evaluation/cases.json\') == cases\n            and read_json(args.output / \'evaluation/generation-config.json\') == GENERATION,\n            \'EVALUATION_INPUT_RECORD_CHANGED\')\n    require({r[\'record_id\'] for r in predictions} == {c[\'record_id\'] for c in cases}, \'EVAL_ID_SET\')\n    for row in predictions:\n        case = next(c for c in cases if c[\'record_id\'] == row[\'record_id\'])\n        require(row[\'prompt_sha256\'] == digest(case[\'prompt\']), \'EVAL_PROMPT_CHANGED\')\n        require(row[\'checks\'] == score_generation(row[\'raw_output\'], case, row[\'status\']), \'CHECK_RECOMPUTATION\')\n        require(math.isfinite(row[\'latency_seconds\']) and row[\'latency_seconds\'] >= 0\n                and type(row[\'output_tokens\']) is int and 0 <= row[\'output_tokens\'] <= 512\n                and type(row[\'input_tokens\']) is int and 0 < row[\'input_tokens\'] <= 1536, \'GENERATION_MEASUREMENTS\')\n    reload = read_json(args.output / \'evaluation/reload.json\')\n    env = read_json(args.output / \'evaluation/environment.json\')\n    require(env[\'cuda_available\'] is True and len(env[\'gpus\']) == 1 and \'T4\' in env[\'gpus\'][0],\n            \'RELOAD_GPU_REQUIRED\')\n    require(reload[\'status\'] == \'GPU_RELOAD_PASSED\' and reload[\'reloaded_saved_tensors_exact\'] is True\n            and math.isfinite(reload[\'forward_loss\']) and reload[\'requires_grad_any\'] is False,\n            \'RELOAD_REPORT\')\n    require(reload[\'model_id\'] == config[\'model_id\'] and reload[\'revision\'] == config[\'revision\']\n            and reload[\'adapter_sha256\'] == sha_file(Path(context[\'pilot\']) / \'final-adapter/adapter_model.safetensors\'),\n            \'RELOAD_ADAPTER_IDENTITY\')\n    resume = audit_resume(args.output)\n    report = dict(status=\'GPU_VERIFICATION_PASSED\', pilot_manifest_sha256=PILOT_SHA,\n                  reload=reload, resume=resume, comparison=comparison,\n                  quality_improvement=\'NOT_ESTABLISHED_UNTIL_CONTENT_REVIEW\')\n    report_path = args.output.with_name(args.output.name + \'-audit.json\')\n    archive = args.output.with_name(args.output.name + \'.zip\')\n    receipt = args.output.with_name(args.output.name + \'-receipt.json\')\n    require(not any(p.exists() for p in (report_path, archive, receipt, args.output / \'manifest.json\')),\n            \'ASSESSMENT_EXPORT_EXISTS\')\n    write_json(args.output / \'summary.json\', report)\n    write_json(args.output / \'manifest.json\', dict(status=report[\'status\'],\n        pilot_manifest_sha256=PILOT_SHA, config_sha256=digest(config),\n        files=inventory(args.output)))\n    write_json(report_path, report)\n    needed = sum(p.stat().st_size for p in args.output.rglob(\'*\') if p.is_file())\n    require(shutil.disk_usage(args.output.parent).free > needed + 256 * 1024 ** 2, \'EXPORT_DISK\')\n    with ZipFile(archive, \'x\', compression=ZIP_DEFLATED) as zipped:\n        for path in sorted(args.output.rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, path.relative_to(args.output).as_posix())\n    write_json(receipt, dict(file=archive.name, sha256=sha_file(archive), size_bytes=archive.stat().st_size,\n        audit_file=report_path.name, audit_sha256=sha_file(report_path),\n        manifest_sha256=sha_file(args.output / \'manifest.json\'), pilot_manifest_sha256=PILOT_SHA,\n        scope=\'PRIVATE_RELOAD_RESUME_DEVELOPMENT_COMPARISON\', base_weights_included=False))\n    print(\'GPU_VERIFICATION_PASSED: \' + str(report_path))\n    print(\'ZIP: \' + str(archive))\n    print(\'Receipt: \' + str(receipt))\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--stage\', choices=(\'prepare\', \'evaluate\', \'export\'), required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--work\', type=Path)\n    parser.add_argument(\'--search\', type=Path, default=Path(\'/kaggle/input\'))\n    parser.add_argument(\'--bundle\', type=Path)\n    parser.add_argument(\'--frozen-tools\', type=Path)\n    parser.add_argument(\'--cache-dir\', type=Path)\n    parser.add_argument(\'--allow-model-download\', action=\'store_true\')\n    args = parser.parse_args(argv)\n    os.environ.setdefault(\'CUDA_VISIBLE_DEVICES\', \'0\')\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\')\n    if args.stage == \'prepare\':\n        require(all(p is not None for p in (args.work, args.bundle, args.frozen_tools)), \'PREPARE_ARGUMENTS\')\n        prepare(args)\n    elif args.stage == \'evaluate\':\n        evaluate(args)\n    else:\n        export(args)\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n'}
VERIFY_SOURCE_ID = 'a47d727976e7325807d011a98162983a43ac1318208dcf6af0937d6f9683b45b'
assert hashlib.sha256(json.dumps(VERIFY_SOURCES, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()).hexdigest() == VERIFY_SOURCE_ID
VERIFY_TOOLS = WORK / ('tools-assessment-' + VERIFY_SOURCE_ID[:12])
if not VERIFY_TOOLS.exists():
    shutil.copytree(RUN_TOOLS, VERIFY_TOOLS)
    for name, content in VERIFY_SOURCES.items():
        (VERIFY_TOOLS / 'scripts/qlora' / name).write_bytes(content.encode())
expected = {p.relative_to(RUN_TOOLS).as_posix() for p in RUN_TOOLS.rglob('*') if p.is_file()} | {'scripts/qlora/' + name for name in VERIFY_SOURCES}
assert {p.relative_to(VERIFY_TOOLS).as_posix() for p in VERIFY_TOOLS.rglob('*') if p.is_file()} == expected
for name in expected:
    content = VERIFY_SOURCES.get(Path(name).name) if name.startswith('scripts/qlora/') else None
    assert (VERIFY_TOOLS / name).read_bytes() == (content.encode() if content is not None else (RUN_TOOLS / name).read_bytes())
FROZEN_SOURCES = {'__init__.py': '"""Isolated QLoRA pilot tooling; never imported by the serving backend."""\n', 'check.py': '"""Run CPU trainer tests and optional real-tokenizer pilot preflight; no Qwen weights."""\nimport argparse\nfrom contextlib import redirect_stderr, redirect_stdout\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport unittest\n\nfrom .data import read_json\nfrom .runtime import environment, finish_manifest, output_guard, write_json\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--bundle\', type=Path)\n    parser.add_argument(\'--config\', type=Path, default=Path(__file__).resolve().parents[2] / \'configs/qlora-pilot.json\')\n    args = parser.parse_args(argv)\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', HF_HUB_OFFLINE=\'1\',\n                      TRANSFORMERS_OFFLINE=\'1\', TOKENIZERS_PARALLELISM=\'false\', WANDB_DISABLED=\'true\')\n    config = read_json(args.config)\n    output = output_guard(args.output, [args.config] + ([args.bundle] if args.bundle else []))\n    output.mkdir(parents=True, exist_ok=False)\n    start = time.monotonic()\n    status = \'FAILED\'\n    error = None\n    try:\n        with (output / \'test-results.txt\').open(\'w\', encoding=\'utf-8\') as stream, \\\n             redirect_stdout(stream), redirect_stderr(stream):\n            suite = unittest.defaultTestLoader.discover(str(Path(__file__).parent / \'tests\'),\n                top_level_dir=str(Path(__file__).resolve().parents[2]))\n            result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)\n            write_json(output / \'tests.json\', {\n                \'tests_run\': result.testsRun, \'failures\': len(result.failures),\n                \'errors\': len(result.errors), \'skipped\': len(result.skipped),\n                \'passed\': result.wasSuccessful(), \'seconds\': time.monotonic() - start,\n                \'scope\': \'CPU unit tests and tiny random Qwen2 LoRA integration, not 7B QLoRA smoke\',\n                \'qwen_7b_weights_loaded\': False, \'gpu_smoke\': \'NOT_RUN\', \'gpu_pilot\': \'NOT_RUN\'})\n            if not result.wasSuccessful():\n                return 1\n            from .tests.test_runtime import RuntimeTests\n            write_json(output / \'tiny-cpu-proof.json\', RuntimeTests.cpu_proof)\n            if args.bundle:\n                from .train import main as preflight\n                preflight([\'--config\', str(args.config), \'--bundle\', str(args.bundle),\n                           \'--profile\', \'smoke\', \'--preflight-only\', \'--output\', str(output / \'pilot-preflight\')])\n            write_json(output / \'environment.json\', environment())\n            status = \'CPU_CHECKS_PASSED\'\n    except Exception as exc:\n        error = type(exc).__name__\n        raise\n    finally:\n        finish_manifest(output, status=status, profile=\'cpu-tests\', config=config, error_type=error)\n    print(status + \': \' + str(output))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'collator.py': '"""Strict right padding for already-audited, unshifted completion-only labels."""\nfrom dataclasses import dataclass\nfrom .data import require\n\n\n@dataclass(frozen=True)\nclass CompletionCollator:\n    pad_token_id: int = 151643\n    eos_token_id: int = 151645\n    max_sequence_tokens: int = 2048\n    pad_to_multiple_of: int = 8\n\n    def pad(self, features):\n        require(bool(features), \'EMPTY_BATCH\')\n        require(type(self.pad_to_multiple_of) is int and self.pad_to_multiple_of > 0,\n                \'PADDING_MULTIPLE\')\n        batch = {k: [] for k in (\'input_ids\', \'attention_mask\', \'labels\')}\n        for row in features:\n            require(set(row) == set(batch), \'MODEL_INPUT_FIELDS\')\n            ids, mask, labels = (row[k] for k in batch)\n            require(all(isinstance(v, list) for v in (ids, mask, labels))\n                    and len(ids) == len(mask) == len(labels) > 0, \'TOKEN_SHAPES\')\n            require(all(type(n) is int and n >= 0 for n in ids)\n                    and all(type(n) is int and n == 1 for n in mask)\n                    and all(type(n) is int for n in labels), \'TOKEN_TYPES_OR_PREPADDING\')\n            positions = [i for i, n in enumerate(labels) if n != -100]\n            require(len(positions) >= 2 and positions[0] > 0\n                    and positions == list(range(positions[0], positions[-1] + 1)),\n                    \'COMPLETION_MASK\')\n            require(all(labels[i] == ids[i] for i in positions)\n                    and labels[positions[-1]] == self.eos_token_id\n                    and sum(labels[i] == self.eos_token_id for i in positions) == 1,\n                    \'LABEL_IDS_OR_EOS\')\n        size = max(len(row[\'input_ids\']) for row in features)\n        size = ((size + self.pad_to_multiple_of - 1) // self.pad_to_multiple_of) * self.pad_to_multiple_of\n        require(size <= self.max_sequence_tokens, \'PADDED_TOKEN_LIMIT\')\n        for row in features:\n            missing = size - len(row[\'input_ids\'])\n            for key, padding in ((\'input_ids\', self.pad_token_id), (\'attention_mask\', 0), (\'labels\', -100)):\n                batch[key].append(row[key][:] + [padding] * missing)\n        return batch\n\n    def __call__(self, features):\n        import torch\n        return {key: torch.tensor(values, dtype=torch.long) for key, values in self.pad(features).items()}\n', 'data.py': '"""Authenticate the locked pilot and recompute its tokenizer boundaries on CPU."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport unicodedata\n\n\ndef require(condition, code):\n    if not condition:\n        raise ValueError(code)\n\n\ndef read_json(path):\n    return loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef loads(text):\n    def pairs(items):\n        result = {}\n        for key, value in items:\n            require(key not in result, \'DUPLICATE_JSON_KEY\')\n            result[key] = value\n        return result\n    def invalid(value):\n        raise ValueError(\'NONFINITE_JSON\')\n    return json.loads(text, object_pairs_hook=pairs, parse_constant=invalid)\n\n\ndef canonical(value):\n    def normalize(item):\n        if isinstance(item, str):\n            return unicodedata.normalize(\'NFC\', item)\n        if isinstance(item, list):\n            return [normalize(v) for v in item]\n        if isinstance(item, dict):\n            return {k: normalize(v) for k, v in item.items()}\n        return item\n    return json.dumps(normalize(value), ensure_ascii=False, sort_keys=True,\n                      separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef sha_file(path):\n    sha = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            sha.update(block)\n    return sha.hexdigest()\n\n\ndef within(root, name):\n    root = Path(root).resolve()\n    require(isinstance(name, str) and \'\\\\\' not in name and \':\' not in name,\n            \'UNSAFE_MEMBER_PATH\')\n    path = (root / name).resolve()\n    require(path.is_relative_to(root) and path != root, \'MEMBER_OUTSIDE_ROOT\')\n    return path\n\n\ndef read_rows(path):\n    rows = []\n    for line in Path(path).read_text(encoding=\'utf-8\').splitlines():\n        require(bool(line.strip()), \'EMPTY_JSONL_ROW\')\n        row = loads(line)\n        require(isinstance(row, dict), \'INVALID_JSONL_ROW\')\n        rows.append(row)\n    return rows\n\n\ndef tokenize_payload(payload, tokenizer, limits):\n    prompt = tokenizer.apply_chat_template(payload[\'prompt\'], tokenize=True,\n                                            add_generation_prompt=True)\n    full = tokenizer.apply_chat_template(payload[\'prompt\'] + payload[\'completion\'],\n                                          tokenize=True, add_generation_prompt=False)\n    require(isinstance(full, list) and isinstance(prompt, list)\n            and all(type(n) is int for n in full + prompt), \'TOKENIZER_IDS\')\n    require(full[:len(prompt)] == prompt, \'CHAT_TEMPLATE_PREFIX\')\n    completion = full[len(prompt):]\n    require(completion.count(tokenizer.eos_token_id) == 1, \'COMPLETION_EOS\')\n    eos = len(prompt) + completion.index(tokenizer.eos_token_id)\n    tail = full[eos + 1:]\n    require(eos > len(prompt), \'EMPTY_COMPLETION\')\n    require(not tail or tail == tokenizer.encode(\'\\n\', add_special_tokens=False), \'POST_EOS_TEXT\')\n    require(len(prompt) <= limits[\'max_prompt_tokens\']\n            and len(completion) <= limits[\'max_completion_tokens\']\n            and len(full) <= limits[\'max_sequence_tokens\'], \'TOKEN_BUDGET\')\n    return dict(input_ids=full, attention_mask=[1] * len(full),\n                labels=[-100] * len(prompt) + full[len(prompt):eos + 1] + [-100] * len(tail),\n                prompt_tokens=len(prompt), completion_tokens=len(completion),\n                sequence_tokens=len(full), eos_position=eos,\n                supervised_tokens=eos + 1 - len(prompt))\n\n\ndef authenticate(folder, config):\n    folder = Path(folder).resolve()\n    manifest_path = folder / \'manifest.json\'\n    require(sha_file(manifest_path) == config[\'data_manifest_sha256\'], \'MANIFEST_HASH\')\n    manifest = read_json(manifest_path)\n    for name, info in manifest[\'files\'].items():\n        path = within(folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'DATA_INTEGRITY:\' + name)\n    contract = read_json(folder / \'training-contract.json\')\n    require(digest(contract) == config[\'contract_sha256\'] == manifest[\'contract_sha256\'],\n            \'CONTRACT_HASH\')\n    require(config[\'model_id\'] == contract[\'model\'][\'model_id\']\n            and config[\'revision\'] == contract[\'model\'][\'revision\']\n            == contract[\'model\'][\'tokenizer_revision\'], \'MODEL_IDENTITY\')\n    selection = read_json(folder / \'selection.json\')\n    require(set(selection[\'splits\']) == {\'train\', \'validation\'}, \'PILOT_SPLITS\')\n    ids = selection[\'splits\'][\'train\'] + selection[\'splits\'][\'validation\']\n    require(len(ids) == len(set(ids)) and not set(ids) & set(selection[\'excluded_holdout_ids\']),\n            \'DUPLICATE_OR_HOLDOUT_ID\')\n    require(manifest[\'counts\'] == {\'train\': 128, \'validation\': 32}\n            and manifest[\'holdout_payload_included\'] is False, \'PILOT_COUNTS\')\n    return contract, manifest, selection\n\n\ndef load_pilot(folder, tokenizer_folder, config):\n    from importlib.metadata import version\n    from transformers import AutoTokenizer\n    folder = Path(folder).resolve()\n    contract, manifest, selection = authenticate(folder, config)\n    reference = read_json(folder / \'tokenizer-reference.json\')\n    for name, expected in reference[\'packages\'].items():\n        require(version(name) == expected, \'TOKENIZER_LIBRARY_VERSION:\' + name)\n    for name, info in reference[\'files\'].items():\n        path = within(tokenizer_folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'TOKENIZER_ASSET:\' + name)\n    tokenizer = AutoTokenizer.from_pretrained(tokenizer_folder, local_files_only=True,\n                                               trust_remote_code=False)\n    require(tokenizer.eos_token_id == contract[\'model\'][\'eos_token_id\']\n            and tokenizer.pad_token_id == contract[\'model\'][\'pad_token_id\']\n            and hashlib.sha256(tokenizer.chat_template.encode()).hexdigest()\n            == contract[\'model\'][\'chat_template_sha256\'], \'TOKENIZER_IDENTITY\')\n    result = {}\n    for split in (\'train\', \'validation\'):\n        tokens = read_rows(folder / (\'tokenized-\' + split + \'.jsonl\'))\n        payloads = read_rows(folder / (split + \'.jsonl\'))\n        metadata = read_rows(folder / (split + \'.metadata.jsonl\'))\n        require(len(tokens) == len(payloads) == len(metadata) == manifest[\'counts\'][split],\n                \'SPLIT_COUNT\')\n        result[split] = []\n        for index, (row, payload, meta) in enumerate(zip(tokens, payloads, metadata)):\n            rid = selection[\'splits\'][split][index]\n            require(row[\'record_id\'] == meta[\'source_record_id\'] == rid\n                    and meta[\'split\'] == split and meta[\'row_index\'] == index, \'ROW_BINDING\')\n            require(digest(payload) == meta[\'training_payload_sha256\'], \'PAYLOAD_HASH\')\n            require(row == dict(record_id=rid, **tokenize_payload(payload, tokenizer,\n                                                                contract[\'token_limits\'])),\n                    \'TOKEN_LABEL_RECOMPUTATION\')\n            # Do not allow IDs or provenance to reach model.forward().\n            result[split].append({k: row[k] for k in (\'input_ids\', \'attention_mask\', \'labels\')})\n    return result, tokenizer, contract\n', 'kaggle.py': '"""Run the versioned trainer in a fresh, separate Kaggle Python environment."""\nimport argparse\nfrom importlib.metadata import version\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nfrom .data import require\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--profile\', choices=(\'smoke\', \'pilot\'), required=True)\n    parser.add_argument(\'--environment\', type=Path, default=Path(\'/kaggle/working/qlora-env\'))\n    parser.add_argument(\'--config\', type=Path, default=Path(__file__).resolve().parents[2] / \'configs/qlora-pilot.json\')\n    parser.add_argument(\'--preflight-only\', action=\'store_true\')\n    parser.add_argument(\'--environment-only\', action=\'store_true\', help=\'Check CUDA, NF4 and data; never train Qwen\')\n    parser.add_argument(\'--allow-model-download\', action=\'store_true\')\n    parser.add_argument(\'--resume-from-checkpoint\', type=Path)\n    parser.add_argument(\'--resume-manifest-sha256\')\n    parser.add_argument(\'--stop-after-step\', type=int)\n    args = parser.parse_args(argv)\n    require(not args.environment_only or (not args.preflight_only and not args.allow_model_download\n            and args.resume_from_checkpoint is None and args.resume_manifest_sha256 is None\n            and args.stop_after_step is None), \'ENVIRONMENT_ONLY_ARGUMENT_CONFLICT\')\n    require(sys.platform == \'linux\' and Path(\'/kaggle/working\').is_dir(), \'KAGGLE_ONLY\')\n    require(args.environment.resolve().is_relative_to(Path(\'/kaggle/working\')),\n            \'ENVIRONMENT_MUST_STAY_IN_KAGGLE_WORKING\')\n    require(not args.output.exists(), \'OUTPUT_ALREADY_EXISTS\')\n    root = Path(__file__).resolve().parents[2]\n    env = dict(os.environ, USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\',\n               PYTHONUTF8=\'1\', PYTHONDONTWRITEBYTECODE=\'1\', WANDB_DISABLED=\'true\')\n    python = args.environment / \'bin/python\'\n    if not args.environment.exists():\n        # Keep the runtime CUDA torch; never replace it with the CPU test wheel.\n        subprocess.run([sys.executable, \'-m\', \'venv\', \'--system-site-packages\',\n                        str(args.environment)], check=True, env=env)\n    require(python.is_file(), \'TRAINING_ENVIRONMENT_INCOMPLETE\')\n    torch_version = version(\'torch\')\n    constraint = args.environment / \'torch-constraint.txt\'\n    constraint.write_text(\'torch==\' + torch_version + \'\\n\', encoding=\'utf-8\')\n    subprocess.run([str(python), \'-m\', \'pip\', \'install\', \'--disable-pip-version-check\',\n                    \'--constraint\', str(constraint),\n                    \'-r\', str(Path(__file__).parent / \'requirements.txt\')], check=True, env=env)\n    installed_torch = subprocess.run([str(python), \'-c\',\n        \'from importlib.metadata import version; print(version("torch"))\'],\n        check=True, capture_output=True, text=True, env=env).stdout.strip()\n    require(installed_torch == torch_version, \'KAGGLE_CUDA_TORCH_CHANGED\')\n    module = \'scripts.qlora.setup\' if args.environment_only else \'scripts.qlora.train\'\n    command = [str(python), \'-B\', \'-m\', module, \'--bundle\', str(args.bundle.resolve()),\n               \'--config\', str(args.config.resolve()), \'--output\', str(args.output.resolve()),\n               *([] if args.environment_only else [\'--profile\', args.profile])]\n    for flag, enabled in ((\'--preflight-only\', args.preflight_only),\n                           (\'--allow-model-download\', args.allow_model_download)):\n        if enabled:\n            command.append(flag)\n    for name in (\'resume_from_checkpoint\', \'resume_manifest_sha256\', \'stop_after_step\'):\n        if getattr(args, name) is not None:\n            command.extend([\'--\' + name.replace(\'_\', \'-\'), str(getattr(args, name))])\n    subprocess.run(command, check=True, env=env, cwd=root)\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'package.py': '"""Package only trainer source, tests, recipe and the single guide for Kaggle."""\nimport argparse\nfrom pathlib import Path\nimport sys\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom .data import sha_file\nfrom .runtime import output_guard, write_json\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New private directory\')\n    parser.add_argument(\'--pilot-archive\', type=Path, help=\'Also build one private Input ZIP for environment setup\')\n    args = parser.parse_args(argv)\n    if args.pilot_archive:\n        from .data import require\n        require(sha_file(args.pilot_archive) == \'2b44e35f7386b506955e9fad29541490c052129a8af6ae0eef7e4f0b7f7c0707\',\n                \'LOCKED_PILOT_ARCHIVE_HASH\')\n    root = Path(__file__).resolve().parents[2]\n    output = output_guard(args.output, [root / \'scripts/qlora\', root / \'configs\', root / \'docs\'])\n    output.mkdir(parents=True, exist_ok=False)\n    paths = sorted((root / \'scripts/qlora\').rglob(\'*.py\')) + [\n        root / \'scripts/qlora/requirements.txt\', root / \'configs/qlora-pilot.json\', root / \'docs/qlora.md\']\n    archive = output / \'qlora-tools.zip\'\n    with ZipFile(archive, \'x\', compression=ZIP_DEFLATED) as zipped:\n        for path in paths:\n            zipped.write(path, path.relative_to(root).as_posix())\n    write_json(output / \'qlora-tools.receipt.json\', {\n        \'file\': archive.name, \'sha256\': sha_file(archive), \'size_bytes\': archive.stat().st_size,\n        \'files\': {p.relative_to(root).as_posix(): sha_file(p) for p in paths},\n        \'data_included\': False, \'model_weights_included\': False,\n        \'gpu_smoke\': \'NOT_RUN\', \'gpu_pilot\': \'NOT_RUN\'})\n    if args.pilot_archive:\n        kit = output / \'qlora-environment-input.zip\'\n        # Inner archives use .bundle so upload services do not unpack them recursively.\n        with ZipFile(kit, \'x\', compression=ZIP_DEFLATED) as zipped:\n            zipped.write(archive, \'tools.bundle\')\n            zipped.write(args.pilot_archive, \'pilot.bundle\')\n        write_json(output / \'qlora-environment-input.receipt.json\', {\n            \'file\': kit.name, \'sha256\': sha_file(kit), \'size_bytes\': kit.stat().st_size,\n            \'inner_archives\': {\'tools.bundle\': sha_file(archive), \'pilot.bundle\': sha_file(args.pilot_archive)},\n            \'scope\': \'PRIVATE_INTERNAL_ENVIRONMENT_SETUP\', \'public_redistribution_allowed\': False,\n            \'qwen_weights_included\': False, \'gpu_check\': \'NOT_RUN\'})\n    print(str(archive))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'pilot.py': '"""Audit and export the fresh 50-step pilot, without loading Qwen or pickle state."""\nimport argparse\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom .data import digest, read_json, read_rows, require, sha_file\nfrom .runtime import validate_recipe, write_json\nfrom .smoke import audit_training\n\n\ndef audit(run):\n    run = Path(run).resolve()\n    report = audit_training(run, profile=\'pilot\', steps=50)\n    config = read_json(run / \'resolved-config.json\')\n    validate_recipe(config, \'pilot\')\n    recipe = Path(__file__).resolve().parents[2] / \'configs/qlora-pilot.json\'\n    require(digest(config) == digest(read_json(recipe)), \'PILOT_RECIPE_IDENTITY\')\n    reference = read_json(run / \'data-reference.json\')\n    require(reference.get(\'resume_checkpoint\') is None\n            and reference.get(\'resume_manifest_sha256\') is None, \'PILOT_MUST_START_FRESH\')\n    args = read_json(run / \'training-arguments.json\')\n    require(args[\'save_steps\'] == args[\'eval_steps\'] == 25\n            and args[\'save_total_limit\'] == 2, \'PILOT_CHECKPOINT_SCHEDULE\')\n    require(all(args.get(key) == value for key, value in config[\'training\'].items())\n            and args.get(\'seed\') == args.get(\'data_seed\') == config[\'seed\']\n            and args.get(\'gradient_checkpointing\') is True, \'PILOT_TRAINING_RECIPE\')\n    evaluations = {r[\'step\']: r[\'eval_loss\'] for r in read_rows(run / \'train-metrics.jsonl\')\n                   if \'eval_loss\' in r}\n    require(set(evaluations) == {25, 50}\n            and all(math.isfinite(v) for v in evaluations.values()), \'PILOT_EVALUATION_STEPS\')\n    require(evaluations[50] == report[\'eval_loss\'], \'PILOT_FINAL_EVAL_MISMATCH\')\n    expected = {\'checkpoint-25\', \'checkpoint-50\'}\n    require({p.name for p in run.glob(\'checkpoint-*\')} == expected, \'PILOT_CHECKPOINT_SET\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\', \'binding.json\'}\n    from safetensors import safe_open\n    import torch\n    for step in (25, 50):\n        folder = run / (\'checkpoint-\' + str(step))\n        require(required <= {p.name for p in folder.iterdir()}, \'PILOT_CHECKPOINT_INCOMPLETE\')\n        require(read_json(folder / \'trainer_state.json\')[\'global_step\'] == step, \'PILOT_CHECKPOINT_STEP\')\n        binding = read_json(folder / \'binding.json\')\n        require(binding[\'config_sha256\'] == digest(config)\n                and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\']\n                and binding[\'global_step\'] == step, \'PILOT_CHECKPOINT_BINDING\')\n        with safe_open(folder / \'adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as tensors:\n            require(bool(tensors.keys()) and all(\'lora_\' in key\n                    and torch.isfinite(tensors.get_tensor(key)).all().item() for key in tensors.keys()),\n                    \'PILOT_CHECKPOINT_TENSORS\')\n        adapter = read_json(folder / \'adapter_config.json\')\n        require(adapter[\'base_model_name_or_path\'] == config[\'model_id\']\n                and adapter[\'revision\'] == config[\'revision\']\n                and all(adapter[key] == config[\'lora\'][key]\n                        for key in (\'r\', \'lora_alpha\', \'lora_dropout\', \'bias\', \'task_type\')),\n                \'PILOT_ADAPTER_IDENTITY\')\n    report.update(checkpoints=sorted(expected), eval_loss_by_step=evaluations,\n                  initialization=\'PINNED_BASE_WITH_FRESH_LORA\', gpu_reload=\'NOT_RUN\',\n                  gpu_resume=\'NOT_RUN\', quality_improvement=\'NOT_ESTABLISHED\')\n    return report\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    run = args.run.resolve()\n    report = audit(run)\n    report_path = run.with_name(run.name + \'-audit.json\')\n    if report_path.exists():\n        require(read_json(report_path) == read_json_value(report), \'PILOT_AUDIT_OUTPUT_DIFFERS\')\n    else:\n        write_json(report_path, report)\n    archive = run.with_name(run.name + \'.zip\')\n    receipt = run.with_name(run.name + \'-receipt.json\')\n    require(not archive.exists() and not receipt.exists(), \'PILOT_EXPORT_EXISTS\')\n    needed = sum(p.stat().st_size for p in run.rglob(\'*\') if p.is_file())\n    require(shutil.disk_usage(run.parent).free >= needed + 256 * 1024 ** 2,\n            \'INSUFFICIENT_SPACE_FOR_PILOT_ZIP\')\n    with ZipFile(archive, \'x\', compression=ZIP_DEFLATED) as zipped:\n        for path in sorted(run.rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, path.relative_to(run).as_posix())\n    write_json(receipt, dict(file=archive.name, sha256=sha_file(archive),\n        size_bytes=archive.stat().st_size, audit_file=report_path.name,\n        audit_sha256=sha_file(report_path), run_manifest_sha256=report[\'run_manifest_sha256\'],\n        model_revision=read_json(run / \'resolved-config.json\')[\'revision\'], global_step=50,\n        scope=\'PRIVATE_GPU_PILOT_HANDOFF\', base_weights_included=False))\n    print(\'GPU_PILOT_PASSED: \' + str(report_path))\n    print(\'Private artifact: \' + str(archive))\n    return 0\n\n\ndef read_json_value(value):\n    # Normalize integer step keys to the representation stored in JSON reports.\n    import json\n    return json.loads(json.dumps(value, allow_nan=False))\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'runtime.py': '"""Trainer setup, update evidence and authenticated resume, independent of serving."""\nfrom datetime import datetime, timezone\nfrom importlib.metadata import distributions, version\nimport json\nimport math\nimport platform\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom .data import digest, read_json, require, sha_file, within\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,\n                                    allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n\n\ndef inventory(folder, excluded=()):\n    folder = Path(folder).resolve()\n    result = {}\n    for path in sorted(folder.rglob(\'*\')):\n        require(not path.is_symlink(), \'ARTIFACT_SYMLINK\')\n        if path.is_file() and path.relative_to(folder).as_posix() not in excluded:\n            result[path.relative_to(folder).as_posix()] = {\n                \'sha256\': sha_file(path), \'size_bytes\': path.stat().st_size}\n    return result\n\n\ndef verify_inventory(folder, files):\n    for name, info in files.items():\n        path = within(folder, name)\n        require(path.is_file() and path.stat().st_size == info[\'size_bytes\']\n                and sha_file(path) == info[\'sha256\'], \'ARTIFACT_INTEGRITY:\' + name)\n\n\ndef validate_recipe(config, profile):\n    require(config[\'schema_version\'] == \'ohayo-qlora-recipe-v1\', \'RECIPE_VERSION\')\n    require(profile in (\'smoke\', \'pilot\'), \'PROFILE\')\n    require(config[\'profiles\'] == {\n        \'smoke\': {\'max_steps\': 2, \'save_steps\': 2, \'eval_steps\': 2},\n        \'pilot\': {\'max_steps\': 50, \'save_steps\': 25, \'eval_steps\': 25}}, \'STEP_PROFILES\')\n    train = config[\'training\']\n    require(train[\'per_device_train_batch_size\'] == train[\'per_device_eval_batch_size\'] == 1\n            and train[\'gradient_accumulation_steps\'] == 4, \'BATCH_RECIPE\')\n    require(all(isinstance(train[k], (float, int)) and not isinstance(train[k], bool)\n                and math.isfinite(train[k]) and train[k] > 0\n                for k in (\'learning_rate\', \'max_grad_norm\'))\n            and 0 <= train[\'warmup_ratio\'] <= 1, \'OPTIMIZER_RECIPE\')\n    require(config[\'seed\'] == 42 and config[\'lora\'] == {\n        \'r\': 8, \'lora_alpha\': 16, \'lora_dropout\': 0.05, \'bias\': \'none\',\n        \'target_modules\': \'all-linear\', \'task_type\': \'CAUSAL_LM\'}, \'LORA_RECIPE\')\n\n\ndef output_guard(output, inputs):\n    output = Path(output).resolve()\n    require(not output.exists(), \'OUTPUT_ALREADY_EXISTS\')\n    for value in inputs:\n        value = Path(value).resolve()\n        require(not output.is_relative_to(value) and not value.is_relative_to(output),\n                \'OUTPUT_OVERLAPS_INPUT\')\n    return output\n\n\ndef training_arguments(config, profile, output, *, cpu_test=False):\n    from transformers import TrainingArguments\n    values = dict(config[\'training\'])\n    args = TrainingArguments(\n        output_dir=str(output), **values, **config[\'profiles\'][profile],\n        seed=config[\'seed\'], data_seed=config[\'seed\'],\n        eval_strategy=\'steps\', save_strategy=\'steps\', logging_strategy=\'steps\', logging_steps=1,\n        save_total_limit=2, save_only_model=False, load_best_model_at_end=False,\n        fp16=not cpu_test, bf16=False, use_cpu=cpu_test,\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={\'use_reentrant\': False},\n        report_to=\'none\', remove_unused_columns=False, label_names=[\'labels\'],\n        dataloader_num_workers=0, dataloader_pin_memory=not cpu_test,\n        disable_tqdm=True, save_safetensors=True, logging_nan_inf_filter=False)\n    if not cpu_test:\n        require(args.n_gpu == 1 and args.world_size == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    return args\n\n\ndef adapter_parameters(model):\n    selected = {}\n    for name, param in model.named_parameters():\n        if param.requires_grad:\n            require(\'lora_\' in name, \'BASE_PARAMETER_TRAINABLE:\' + name)\n            selected[name] = param\n    require(bool(selected), \'NO_TRAINABLE_ADAPTER\')\n    return selected\n\n\ndef parameter_report(model):\n    selected = adapter_parameters(model)\n    return dict(trainable_parameters=sum(p.numel() for p in selected.values()),\n                total_parameter_elements=sum(p.numel() for p in model.parameters()),\n                trainable_names=sorted(selected), base_frozen=True,\n                note=\'Quantized parameter elements are storage counts, not dense base parameter counts.\')\n\n\ndef resume_checkpoint(checkpoint, expected_manifest_sha256, config, stop_after_step):\n    checkpoint = Path(checkpoint).resolve()\n    parent = checkpoint.parent\n    require(expected_manifest_sha256 is not None, \'RESUME_RECEIPT_REQUIRED\')\n    require(sha_file(parent / \'run-manifest.json\') == expected_manifest_sha256,\n            \'RESUME_MANIFEST_HASH\')\n    report = read_json(parent / \'run-manifest.json\')\n    require(report[\'status\'] == \'TRAINING_COMPLETED\' and report[\'profile\'] == \'pilot\'\n            and report[\'config_sha256\'] == digest(config), \'RESUME_RUN_IDENTITY\')\n    verify_inventory(parent, report[\'files\'])\n    require(set(inventory(parent, (\'run-manifest.json\', \'SHA256SUMS.txt\')))\n            == set(report[\'files\']), \'RESUME_EXTRA_FILES\')\n    code = {p.name: sha_file(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}\n    require(report[\'source_files_sha256\'] == code, \'RESUME_TRAINER_CODE_CHANGED\')\n    require(checkpoint.name == \'checkpoint-25\' and checkpoint.is_dir(), \'RESUME_STEP_25_REQUIRED\')\n    binding = read_json(checkpoint / \'binding.json\')\n    require(binding[\'config_sha256\'] == digest(config)\n            and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\'], \'RESUME_BINDING\')\n    require(read_json(checkpoint / \'trainer_state.json\')[\'global_step\'] == 25,\n            \'RESUME_STATE_STEP\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\'}\n    require(required <= {p.name for p in checkpoint.iterdir()}, \'INCOMPLETE_RESUME_STATE\')\n    require(stop_after_step == 26, \'RESUME_STOP_26_REQUIRED\')\n    return checkpoint\n\n\ndef make_callbacks(output, config, *, stop_after_step=None):\n    import torch\n    from transformers import TrainerCallback\n\n    class EvidenceCallback(TrainerCallback):\n        trainer = None\n\n        def __init__(self):\n            self.before = {}\n            self.applied = 0\n            self.skipped = 0\n            self.gradient_checks = 0\n            self.started = None\n            self.start_step = None\n\n        def on_train_begin(self, args, state, control, model=None, **kwargs):\n            self.started = time.monotonic()\n            self.start_step = state.global_step\n            self.before = {n: p.detach().float().cpu().clone()\n                           for n, p in adapter_parameters(model).items()}\n\n        def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):\n            gradients = [p.grad for p in adapter_parameters(model).values() if p.grad is not None]\n            require(bool(gradients), \'NO_ADAPTER_GRADIENT\')\n            require(all(torch.isfinite(g).all().item() for g in gradients), \'NONFINITE_GRADIENT\')\n            self.gradient_checks += 1\n\n        def on_step_end(self, args, state, control, **kwargs):\n            skipped = bool(self.trainer.accelerator.optimizer_step_was_skipped)\n            self.skipped += int(skipped)\n            self.applied += int(not skipped)\n            event = dict(step=state.global_step, update_applied=not skipped,\n                         elapsed_seconds=time.monotonic() - self.started)\n            with (output / \'optimizer-events.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(event, allow_nan=False) + \'\\n\')\n            write_json(output / \'phase.json\', dict(phase=\'TRAINING_AND_VALIDATION\', **event))\n            if stop_after_step is not None and state.global_step >= stop_after_step:\n                control.should_training_stop = True\n                control.should_save = True\n                control.should_evaluate = True\n            return control\n\n        def on_log(self, args, state, control, logs=None, **kwargs):\n            logs = dict(logs or {})\n            for value in logs.values():\n                if isinstance(value, (int, float)):\n                    require(math.isfinite(value), \'NONFINITE_TRAINING_LOG\')\n            with (output / \'train-metrics.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(dict(step=state.global_step, **logs), allow_nan=False) + \'\\n\')\n\n        def on_save(self, args, state, control, **kwargs):\n            checkpoint = output / (\'checkpoint-\' + str(state.global_step))\n            write_json(checkpoint / \'binding.json\', {\n                \'config_sha256\': digest(config), \'data_manifest_sha256\': config[\'data_manifest_sha256\'],\n                \'global_step\': state.global_step})\n\n        def report(self, model, global_step):\n            after = adapter_parameters(model)\n            require(set(after) == set(self.before), \'ADAPTER_PARAMETERS_CHANGED\')\n            squared = sum((p.detach().float().cpu() - self.before[n]).double().square().sum().item()\n                          for n, p in after.items())\n            return dict(start_global_step=self.start_step, global_step=global_step,\n                        attempted_updates=self.applied + self.skipped,\n                        applied_updates=self.applied, scaler_skipped_updates=self.skipped,\n                        finite_gradient_checks=self.gradient_checks,\n                        adapter_delta_l2=math.sqrt(squared), base_frozen=True)\n\n    return EvidenceCallback()\n\n\ndef fit(model, data, collator, args, output, config, *, resume=None, stop_after_step=None):\n    from transformers import Trainer\n    if not args.use_cpu:\n        require(args.n_gpu == 1 and args.world_size == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    evidence = make_callbacks(output, config, stop_after_step=stop_after_step)\n    trainer = Trainer(model=model, args=args, train_dataset=data[\'train\'],\n                      eval_dataset=data[\'validation\'], data_collator=collator,\n                      callbacks=[evidence])\n    evidence.trainer = trainer\n    try:\n        trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    except BaseException as exc:\n        # A CUDA failure can poison the context. Do not copy tensors back to CPU\n        # while handling it, or replace the original exception with a second one.\n        failure = dict(error_type=type(exc).__name__, error=str(exc),\n                       global_step=trainer.state.global_step,\n                       observed_applied_updates=evidence.applied,\n                       observed_skipped_updates=evidence.skipped,\n                       adapter_delta_l2=None, tensor_measurement=\'NOT_ATTEMPTED_ON_FAILED_RUN\')\n        try:\n            write_json(output / \'training-failure.json\', failure)\n        except OSError:\n            pass\n        raise\n    update = evidence.report(model, trainer.state.global_step)\n    write_json(output / \'adapter-update-report.json\', update)\n    target = stop_after_step or args.max_steps\n    require(trainer.state.global_step == target and update[\'applied_updates\'] > 0\n            and update[\'scaler_skipped_updates\'] == 0\n            and update[\'attempted_updates\'] == target - update[\'start_global_step\']\n            and update[\'finite_gradient_checks\'] == update[\'attempted_updates\']\n            and math.isfinite(update[\'adapter_delta_l2\']) and update[\'adapter_delta_l2\'] > 0,\n            \'OPTIMIZER_UPDATE_PROOF_FAILED\')\n    metrics = next((entry for entry in reversed(trainer.state.log_history)\n                    if entry.get(\'step\') == trainer.state.global_step and \'eval_loss\' in entry), None)\n    if metrics is None:\n        metrics = trainer.evaluate()\n    require(math.isfinite(metrics[\'eval_loss\']), \'NONFINITE_EVAL_LOSS\')\n    write_json(output / \'eval-metrics.json\', metrics)\n    trainer.save_model(str(output / \'final-adapter\'))\n    trainer.save_state()\n    return trainer, update\n\n\ndef environment():\n    import torch\n    return dict(python=sys.version, platform=platform.platform(), torch=torch.__version__,\n                cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(),\n                gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                dependencies={d.metadata[\'Name\']: d.version for d in distributions()},\n                recorded_at=datetime.now(timezone.utc).isoformat())\n\n\ndef load_training_model(config, *, allow_download=False, cache_dir=None):\n    import torch\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    require(torch.cuda.is_available(), \'CUDA_REQUIRED\')\n    require(torch.cuda.device_count() == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    require(\'T4\' in torch.cuda.get_device_name(0), \'PROFILE_REQUIRES_T4\')\n    require(not torch.distributed.is_initialized(), \'SINGLE_PROCESS_ONLY\')\n    free, total = torch.cuda.mem_get_info(0)\n    require(free >= 12 * 1024 ** 3, \'GPU_BUSY_OR_INSUFFICIENT_FREE_VRAM\')\n    for name, expected in config[\'dependencies\'].items():\n        require(version(name) == expected, \'DEPENDENCY_VERSION:\' + name)\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                      bnb_4bit_use_double_quant=True,\n                                      bnb_4bit_compute_dtype=torch.float16)\n    model = AutoModelForCausalLM.from_pretrained(\n        config[\'model_id\'], revision=config[\'revision\'], trust_remote_code=False,\n        local_files_only=not allow_download, quantization_config=quantization,\n        torch_dtype=torch.float16, device_map={\'\': 0}, attn_implementation=\'sdpa\',\n        use_safetensors=True, cache_dir=str(cache_dir) if cache_dir is not None else None)\n    require(model.config._commit_hash == config[\'revision\'], \'BASE_REVISION_MISMATCH\')\n    require(getattr(model, \'is_loaded_in_4bit\', False), \'BASE_NOT_4BIT\')\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,\n                                            gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    model = get_peft_model(model, LoraConfig(**config[\'lora\'], revision=config[\'revision\']))\n    adapter_parameters(model)\n    return model\n\n\ndef finish_manifest(output, *, status, profile, config, error_type=None):\n    source = Path(__file__).parent\n    code = {p.name: sha_file(p) for p in sorted(source.glob(\'*.py\'))}\n    try:\n        git = subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], capture_output=True, text=True,\n                             check=True, cwd=source).stdout.strip()\n    except (OSError, subprocess.SubprocessError):\n        git = None\n    manifest = dict(status=status, profile=profile, config_sha256=digest(config),\n                    data_manifest_sha256=config[\'data_manifest_sha256\'],\n                    source_files_sha256=code, git_head=git,\n                    note=\'Source file hashes identify uncommitted code; git_head alone does not.\',\n                    error_type=error_type,\n                    files=inventory(output, (\'run-manifest.json\', \'SHA256SUMS.txt\')))\n    write_json(output / \'run-manifest.json\', manifest)\n    files = inventory(output, (\'SHA256SUMS.txt\',))\n    (output / \'SHA256SUMS.txt\').write_text(\'\'.join(\n        info[\'sha256\'] + \'  \' + name + \'\\n\' for name, info in files.items()), encoding=\'utf-8\')\n', 'setup.py': '"""GPU environment checks only: no Qwen weights, LoRA training or optimizer steps."""\nimport argparse\nimport gc\nfrom importlib.metadata import version\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom contextlib import redirect_stderr, redirect_stdout\n\nfrom .collator import CompletionCollator\nfrom .data import load_pilot, read_json, require\nfrom .runtime import environment, finish_manifest, output_guard, write_json\n\n\ndef check_hardware(torch, working):\n    require(torch.cuda.is_available(), \'CUDA_REQUIRED\')\n    require(torch.cuda.device_count() == 1, \'SINGLE_VISIBLE_GPU_REQUIRED_NO_DATAPARALLEL\')\n    require(torch.version.cuda is not None, \'CUDA_TORCH_REQUIRED_NOT_CPU_WHEEL\')\n    require(\'T4\' in torch.cuda.get_device_name(0), \'PROFILE_REQUIRES_T4\')\n    free, total = torch.cuda.mem_get_info(0)\n    require(free >= 12 * 1024 ** 3, \'GPU_BUSY_OR_INSUFFICIENT_FREE_VRAM\')\n    disk = shutil.disk_usage(working)\n    # This phase writes small reports/tensors only; base-cache budgeting belongs\n    # to the weight-loading phase and must not be reported as verified here.\n    require(disk.free >= 1024 ** 3, \'NEED_1_GIB_FREE_FOR_ENVIRONMENT_CHECK\')\n    return dict(device=torch.cuda.get_device_name(0), free_vram_bytes=free,\n                total_vram_bytes=total, free_working_disk_bytes=disk.free,\n                free_working_disk_gib=round(disk.free / 1024 ** 3, 3),\n                training_storage_check=\'PENDING_BEFORE_BASE_WEIGHT_LOADING\')\n\n\ndef probe_nf4(torch):\n    import bitsandbytes as bnb\n    # A random frozen layer tests CUDA kernels, not Qwen or an optimizer update.\n    torch.manual_seed(42)\n    with torch.cuda.device(0):\n        layer = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=torch.float16,\n                                  compress_statistics=True, quant_type=\'nf4\')\n        layer.requires_grad_(False)\n        layer = layer.to(\'cuda:0\')\n        sample = torch.randn(2, 64, device=\'cuda:0\', dtype=torch.float16, requires_grad=True)\n        output = layer(sample)\n        require(output.dtype == torch.float16 and torch.isfinite(output).all().item(), \'NF4_FP16_FORWARD\')\n        output.float().square().mean().backward()\n        require(sample.grad is not None and torch.isfinite(sample.grad).all().item()\n                and sample.grad.abs().sum().item() > 0, \'NF4_INPUT_GRADIENT\')\n        state = layer.weight.quant_state\n        require(layer.weight.device.type == \'cuda\' and state.quant_type == \'nf4\'\n                and state.nested, \'NF4_DOUBLE_QUANTIZATION\')\n        torch.cuda.synchronize(0)\n        del sample, output, layer\n    gc.collect()\n    torch.cuda.empty_cache()\n    return dict(nf4_cuda_forward=\'PASSED\', fp16_compute=\'PASSED\', input_backward=\'PASSED\',\n                double_quantization=True, optimizer_steps=0, qwen_weights_loaded=False)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', type=Path, required=True)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\',\n                      PYTHONDONTWRITEBYTECODE=\'1\', WANDB_DISABLED=\'true\')\n    config = read_json(args.config)\n    output = output_guard(args.output, [args.bundle, args.config])\n    output.mkdir(parents=True, exist_ok=False)\n    status, error = \'FAILED\', None\n    try:\n        with (output / \'setup-output.txt\').open(\'w\', encoding=\'utf-8\') as stream, \\\n             redirect_stdout(stream), redirect_stderr(stream):\n            import torch\n            from packaging.version import Version\n            write_json(output / \'environment.json\', environment())\n            require(Version(torch.__version__.split(\'+\')[0]) >= Version(\'2.6\'), \'TORCH_2_6_OR_NEWER_REQUIRED\')\n            for name, expected in config[\'dependencies\'].items():\n                require(version(name) == expected, \'DEPENDENCY_VERSION:\' + name)\n            hardware = check_hardware(torch, output)\n            from transformers import Trainer, TrainingArguments\n            from peft import LoraConfig, prepare_model_for_kbit_training\n            from accelerate import Accelerator\n            probe = probe_nf4(torch)\n            data, tokenizer, contract = load_pilot(args.bundle / \'data/pilot-v2\',\n                                                   args.bundle / \'tokenizer\', config)\n            collator = CompletionCollator()\n            for rows in data.values():\n                for offset in range(0, len(rows), 4):\n                    collator(rows[offset:offset + 4])\n            write_json(output / \'environment-check.json\', dict(\n                status=\'ENVIRONMENT_READY_FOR_SMOKE\', hardware=hardware, kernel_probe=probe,\n                records_checked=sum(map(len, data.values())), train=len(data[\'train\']),\n                validation=len(data[\'validation\']), qwen_weights_loaded=False,\n                qwen_training=\'NOT_RUN\', gpu_smoke=\'NOT_RUN\', gpu_pilot=\'NOT_RUN\'))\n            status = \'ENVIRONMENT_READY_FOR_SMOKE\'\n    except BaseException as exc:\n        error = type(exc).__name__\n        write_json(output / \'environment-check.json\', dict(status=\'FAILED\', error_type=error,\n                   error=str(exc), qwen_training=\'NOT_RUN\', gpu_smoke=\'NOT_RUN\', gpu_pilot=\'NOT_RUN\'))\n        raise\n    finally:\n        finish_manifest(output, status=status, profile=\'environment-only\', config=config, error_type=error)\n    print(status + \': \' + str(output))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'smoke.py': '"""Audit and package a completed two-step GPU smoke, without loading Qwen again."""\nimport argparse\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom .data import digest, read_json, read_rows, require, sha_file\nfrom .runtime import inventory, verify_inventory, write_json\n\n\ndef audit(run):\n    return audit_training(run, profile=\'smoke\', steps=2)\n\n\ndef audit_training(run, *, profile, steps):\n    run = Path(run).resolve()\n    manifest = read_json(run / \'run-manifest.json\')\n    require(manifest[\'status\'] == \'TRAINING_COMPLETED\' and manifest[\'profile\'] == profile,\n            \'NOT_COMPLETED_GPU_SMOKE\')\n    verify_inventory(run, manifest[\'files\'])\n    require(set(inventory(run, (\'run-manifest.json\', \'SHA256SUMS.txt\'))) == set(manifest[\'files\']),\n            \'SMOKE_UNLISTED_FILES\')\n    sums = {}\n    for line in (run / \'SHA256SUMS.txt\').read_text(encoding=\'utf-8\').splitlines():\n        sha, name = line.split(\'  \', 1)\n        require(name not in sums, \'SMOKE_DUPLICATE_CHECKSUM\')\n        sums[name] = sha\n    require(set(sums) == set(manifest[\'files\']) | {\'run-manifest.json\'}\n            and all(sha_file(run / name) == sha for name, sha in sums.items()), \'SMOKE_CHECKSUM_LIST\')\n    config = read_json(run / \'resolved-config.json\')\n    require(digest(config) == manifest[\'config_sha256\'], \'SMOKE_CONFIG_HASH\')\n    reference = read_json(run / \'data-reference.json\')\n    require(reference[\'manifest_sha256\'] == config[\'data_manifest_sha256\']\n            == manifest[\'data_manifest_sha256\']\n            and reference[\'contract_sha256\'] == config[\'contract_sha256\']\n            and (reference[\'train\'], reference[\'validation\'], reference[\'holdout\']) == (128, 32, 0),\n            \'SMOKE_DATA_IDENTITY\')\n    args = read_json(run / \'training-arguments.json\')\n    require(args[\'max_steps\'] == steps and args[\'fp16\'] is True and args[\'bf16\'] is False\n            and args[\'gradient_accumulation_steps\'] == 4\n            and args[\'per_device_train_batch_size\'] == 1, \'SMOKE_TRAINING_ARGUMENTS\')\n    env = read_json(run / \'environment.json\')\n    require(env[\'cuda_available\'] is True and env[\'cuda\'] is not None\n            and len(env[\'gpus\']) == 1 and \'T4\' in env[\'gpus\'][0], \'SMOKE_GPU_EVIDENCE_REQUIRED\')\n    devices = read_json(run / \'training-device.json\')\n    require(devices[\'visible_gpu_count\'] == devices[\'world_size\'] == 1\n            and devices[\'mode\'] == \'SINGLE_GPU_NO_DATAPARALLEL\', \'SMOKE_SINGLE_GPU_REQUIRED\')\n    loaded = read_json(run / \'model-load-report.json\')\n    require(loaded[\'model_id\'] == config[\'model_id\'] and loaded[\'revision\'] == config[\'revision\']\n            and loaded[\'is_loaded_in_4bit\'] is True and loaded[\'use_cache\'] is False\n            and loaded[\'quantization\'][\'bnb_4bit_quant_type\'] == \'nf4\'\n            and loaded[\'quantization\'][\'bnb_4bit_use_double_quant\'] is True, \'SMOKE_BASE_LOAD_IDENTITY\')\n    update = read_json(run / \'adapter-update-report.json\')\n    require((update[\'start_global_step\'], update[\'global_step\'], update[\'attempted_updates\'],\n             update[\'applied_updates\'], update[\'scaler_skipped_updates\'], update[\'finite_gradient_checks\'])\n            == (0, steps, steps, steps, 0, steps) and math.isfinite(update[\'adapter_delta_l2\'])\n            and update[\'adapter_delta_l2\'] > 0 and update[\'base_frozen\'] is True,\n            \'SMOKE_OPTIMIZER_PROOF\')\n    events = read_rows(run / \'optimizer-events.jsonl\')\n    require([r[\'step\'] for r in events] == list(range(1, steps + 1))\n            and all(r[\'update_applied\'] is True for r in events), \'SMOKE_UPDATE_EVENTS\')\n    logs = read_rows(run / \'train-metrics.jsonl\')\n    loss = {r[\'step\']: r[\'loss\'] for r in logs if \'loss\' in r}\n    require(set(loss) == set(range(1, steps + 1)) and all(math.isfinite(v) for v in loss.values()), \'SMOKE_FINITE_LOSS\')\n    evaluation = read_json(run / \'eval-metrics.json\')\n    require(math.isfinite(evaluation[\'eval_loss\']), \'SMOKE_FINITE_EVAL_LOSS\')\n    require(read_json(run / \'storage-check.json\')[\'passed\'] is True, \'SMOKE_STORAGE_CHECK\')\n    parameters = read_json(run / \'trainable-parameters.json\')\n    require(parameters[\'base_frozen\'] is True and parameters[\'trainable_parameters\'] > 0,\n            \'SMOKE_BASE_FREEZE\')\n    required = {\'adapter_config.json\', \'adapter_model.safetensors\', \'trainer_state.json\',\n                \'optimizer.pt\', \'scheduler.pt\', \'rng_state.pth\', \'scaler.pt\', \'binding.json\'}\n    checkpoint = run / (\'checkpoint-\' + str(steps))\n    require(checkpoint.is_dir() and required <= {p.name for p in checkpoint.iterdir()},\n            \'SMOKE_CHECKPOINT_INCOMPLETE\')\n    require(read_json(checkpoint / \'trainer_state.json\')[\'global_step\'] == steps,\n            \'SMOKE_CHECKPOINT_STEP\')\n    binding = read_json(checkpoint / \'binding.json\')\n    require(binding[\'config_sha256\'] == digest(config)\n            and binding[\'data_manifest_sha256\'] == config[\'data_manifest_sha256\'], \'SMOKE_CHECKPOINT_BINDING\')\n    from safetensors import safe_open\n    import torch\n    with safe_open(run / \'final-adapter/adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as final, \\\n         safe_open(checkpoint / \'adapter_model.safetensors\', framework=\'pt\', device=\'cpu\') as saved:\n        require(set(final.keys()) == set(saved.keys()) and bool(final.keys()), \'SMOKE_SAVED_TENSOR_KEYS\')\n        count = 0\n        for name in final.keys():\n            require(\'lora_\' in name, \'SMOKE_UNEXPECTED_ADAPTER_TENSOR\')\n            tensor = final.get_tensor(name)\n            require(torch.isfinite(tensor).all().item() and torch.equal(tensor, saved.get_tensor(name)),\n                    \'SMOKE_SAVED_TENSORS_DIFFER\')\n            count += tensor.numel()\n    require(count == parameters[\'trainable_parameters\'], \'SMOKE_PARAMETER_COUNT\')\n    vram = read_json(run / \'vram.json\')\n    require(0 < vram[\'peak_allocated_bytes\'] <= vram[\'peak_reserved_bytes\'], \'SMOKE_VRAM_METRICS\')\n    return dict(status=\'GPU_SMOKE_PASSED\' if profile == \'smoke\' else \'GPU_PILOT_PASSED\',\n                global_step=steps, applied_updates=steps,\n                scaler_skipped_updates=0, adapter_delta_l2=update[\'adapter_delta_l2\'], base_frozen=True,\n                train_loss_by_step=loss, eval_loss=evaluation[\'eval_loss\'],\n                trainable_parameters=count, checkpoint=checkpoint.name, saved_adapter_tensors_match=True,\n                vram=vram, config_sha256=digest(config), data_manifest_sha256=config[\'data_manifest_sha256\'],\n                run_manifest_sha256=sha_file(run / \'run-manifest.json\'),\n                limitation=(\'Two-step technical smoke, not pilot 50, reload/resume verification or quality improvement.\'\n                            if profile == \'smoke\' else\n                            \'Technical pilot 50, not independent GPU reload/resume verification or quality improvement.\'))\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    run = args.run.resolve()\n    report = audit(run)\n    report_path = run.with_name(run.name + \'-audit.json\')\n    if report_path.exists():\n        require(read_json(report_path) == report, \'SMOKE_AUDIT_OUTPUT_DIFFERS\')\n    else:\n        write_json(report_path, report)\n    archive = run.with_name(run.name + \'.zip\')\n    require(not archive.exists(), \'SMOKE_ZIP_EXISTS\')\n    needed = sum(p.stat().st_size for p in run.rglob(\'*\') if p.is_file())\n    require(shutil.disk_usage(run.parent).free >= needed + 256 * 1024 ** 2, \'INSUFFICIENT_SPACE_FOR_SMOKE_ZIP\')\n    with ZipFile(archive, \'x\', compression=ZIP_DEFLATED) as zipped:\n        for path in sorted(run.rglob(\'*\')):\n            if path.is_file():\n                zipped.write(path, path.relative_to(run).as_posix())\n    write_json(run.with_name(run.name + \'-receipt.json\'), dict(\n        file=archive.name, sha256=sha_file(archive), size_bytes=archive.stat().st_size,\n        audit_file=report_path.name, audit_sha256=sha_file(report_path),\n        run_manifest_sha256=report[\'run_manifest_sha256\'], model_revision=read_json(run / \'resolved-config.json\')[\'revision\'],\n        global_step=2, scope=\'PRIVATE_GPU_SMOKE_HANDOFF\', base_weights_included=False))\n    print(\'GPU_SMOKE_PASSED: \' + str(report_path))\n    print(\'Private artifact: \' + str(archive))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', 'storage.py': '"""Budget pinned safetensors downloads and run artifacts before loading any weights."""\nfrom pathlib import Path\nimport re\nimport shutil\n\nfrom .data import read_json, require\n\nGIB = 1024 ** 3\n\n\ndef budget(missing_bytes, cache_free, output_free, same_volume):\n    require(type(missing_bytes) is int and missing_bytes >= 0, \'INVALID_WEIGHT_SIZE\')\n    cache_required = missing_bytes + (GIB if missing_bytes else 0)\n    output_required = 2 * GIB\n    if same_volume:\n        passed = min(cache_free, output_free) >= cache_required + output_required\n    else:\n        passed = cache_free >= cache_required and output_free >= output_required\n    return dict(passed=passed, missing_weight_bytes=missing_bytes,\n                cache_required_bytes=cache_required, output_required_bytes=output_required,\n                cache_free_bytes=cache_free, output_free_bytes=output_free,\n                shared_volume=same_volume,\n                note=\'Reserve 1 GiB for download overhead when needed and 2 GiB for run/checkpoint/ZIP; no base copy in the ZIP.\')\n\n\ndef check_training_storage(config, cache_dir, output, *, allow_download=False):\n    from huggingface_hub import HfApi, hf_hub_download, try_to_load_from_cache\n    cache = Path(cache_dir).expanduser().resolve()\n    output = Path(output).resolve()\n    cache.mkdir(parents=True, exist_ok=True)\n    identity = dict(repo_id=config[\'model_id\'], revision=config[\'revision\'], cache_dir=str(cache))\n    if allow_download:\n        # Metadata only: no safetensors are downloaded until the budget passes.\n        info = HfApi().model_info(config[\'model_id\'], revision=config[\'revision\'],\n                                  files_metadata=True, timeout=30)\n        require(info.sha == config[\'revision\'], \'WEIGHT_METADATA_REVISION\')\n        weights = {item.rfilename: item.size for item in info.siblings\n                   if re.fullmatch(r\'model(?:-\\d+-of-\\d+)?\\.safetensors\', item.rfilename)}\n        require(0 < len(weights) <= 16 and all(type(n) is int and n > 0 for n in weights.values()),\n                \'SAFETENSORS_SIZE_METADATA_REQUIRED\')\n    else:\n        index = hf_hub_download(filename=\'model.safetensors.index.json\',\n                                local_files_only=True, **identity)\n        names = set(read_json(index)[\'weight_map\'].values())\n        weights = {}\n        for name in names:\n            require(re.fullmatch(r\'model-\\d+-of-\\d+\\.safetensors\', name) is not None,\n                    \'WEIGHT_INDEX_FILENAME\')\n            cached = try_to_load_from_cache(filename=name, **identity)\n            require(isinstance(cached, str) and Path(cached).is_file(), \'BASE_NOT_FULLY_CACHED\')\n            weights[name] = Path(cached).stat().st_size\n    cached_bytes = 0\n    for name, size in weights.items():\n        cached = try_to_load_from_cache(filename=name, **identity)\n        if isinstance(cached, str) and Path(cached).is_file() and Path(cached).stat().st_size == size:\n            cached_bytes += size\n    total = sum(weights.values())\n    result = budget(total - cached_bytes, shutil.disk_usage(cache).free,\n                    shutil.disk_usage(output).free, cache.stat().st_dev == output.stat().st_dev)\n    return dict(result, cache_dir=str(cache), model_id=config[\'model_id\'], revision=config[\'revision\'],\n                total_weight_bytes=total, cached_weight_bytes=cached_bytes, weight_files=weights,\n                weights_downloaded_by_check=False, offline=not allow_download)\n', 'train.py': '"""CLI: CPU preflight, separate T4 smoke/pilot runs, and authenticated resume."""\nimport argparse\nfrom contextlib import redirect_stderr, redirect_stdout\nimport os\nfrom pathlib import Path\nimport sys\n\nfrom .collator import CompletionCollator\nfrom .data import load_pilot, read_json, require\nfrom .runtime import (environment, finish_manifest, fit, load_training_model, output_guard,\n                      parameter_report, resume_checkpoint, training_arguments, validate_recipe, write_json)\nfrom .storage import check_training_storage\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--bundle\', type=Path, required=True, help=\'Extracted immutable handoff root\')\n    parser.add_argument(\'--profile\', choices=(\'smoke\', \'pilot\'), required=True)\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New private run directory\')\n    parser.add_argument(\'--preflight-only\', action=\'store_true\')\n    parser.add_argument(\'--allow-model-download\', action=\'store_true\')\n    parser.add_argument(\'--cache-dir\', type=Path)\n    parser.add_argument(\'--resume-from-checkpoint\', type=Path)\n    parser.add_argument(\'--resume-manifest-sha256\')\n    parser.add_argument(\'--stop-after-step\', type=int)\n    args = parser.parse_args(argv)\n    os.environ.update(USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\', WANDB_DISABLED=\'true\')\n    os.environ.setdefault(\'CUDA_VISIBLE_DEVICES\', \'0\')\n    config = read_json(args.config)\n    validate_recipe(config, args.profile)\n    inputs = [args.bundle, args.config]\n    if args.resume_from_checkpoint:\n        inputs.append(args.resume_from_checkpoint.parent)\n    output = output_guard(args.output, inputs)\n    require(args.resume_from_checkpoint is not None or (args.stop_after_step is None\n            and args.resume_manifest_sha256 is None), \'RESUME_ARGUMENTS_REQUIRED\')\n    require(not args.preflight_only or args.resume_from_checkpoint is None, \'PREFLIGHT_NOT_RESUME\')\n    resume = None\n    if args.resume_from_checkpoint:\n        require(args.profile == \'pilot\', \'RESUME_PILOT_ONLY\')\n        resume = resume_checkpoint(args.resume_from_checkpoint, args.resume_manifest_sha256,\n                                    config, args.stop_after_step)\n    output.mkdir(parents=True, exist_ok=False)\n    status, error = \'FAILED\', None\n    def phase(name):\n        write_json(output / \'phase.json\', {\'phase\': name, \'profile\': args.profile})\n    try:\n        phase(\'VALIDATING_DATA\')\n        with (output / \'stdout.txt\').open(\'w\', encoding=\'utf-8\') as stdout, \\\n             (output / \'stderr.txt\').open(\'w\', encoding=\'utf-8\') as stderr, \\\n             redirect_stdout(stdout), redirect_stderr(stderr):\n            import torch\n            from transformers import set_seed\n            set_seed(config[\'seed\'])\n            write_json(output / \'resolved-config.json\', config)\n            source = output / \'source/qlora\'\n            source.mkdir(parents=True)\n            for code in Path(__file__).parent.glob(\'*.py\'):\n                (source / code.name).write_bytes(code.read_bytes())\n            actual_environment = environment()\n            write_json(output / \'environment.json\', actual_environment)\n            (output / \'requirements-resolved.txt\').write_text(\'\'.join(\n                name + \'==\' + ver + \'\\n\' for name, ver in sorted(\n                    actual_environment[\'dependencies\'].items())), encoding=\'utf-8\')\n            write_json(output / \'data-reference.json\', {\n                \'manifest_sha256\': config[\'data_manifest_sha256\'],\n                \'contract_sha256\': config[\'contract_sha256\'],\n                \'train\': 128, \'validation\': 32, \'holdout\': 0,\n                \'resume_checkpoint\': str(resume) if resume else None,\n                \'resume_manifest_sha256\': args.resume_manifest_sha256})\n            data, tokenizer, contract = load_pilot(args.bundle / \'data/pilot-v2\',\n                                                   args.bundle / \'tokenizer\', config)\n            collator = CompletionCollator()\n            shapes = []\n            for split, rows in data.items():\n                for start in range(0, len(rows), 4):\n                    batch = collator(rows[start:start + 4])\n                    shapes.append(list(batch[\'input_ids\'].shape))\n            write_json(output / \'preflight.json\', {\n                \'passed\': True, \'records_checked\': sum(map(len, data.values())),\n                \'batch_shapes\': shapes, \'tensor_dtype\': \'torch.int64\',\n                \'tokenizer_recomputed\': True, \'model_weights_loaded\': False,\n                \'training\': \'NOT_RUN\'})\n            if args.preflight_only:\n                status = \'PREFLIGHT_PASSED\'\n            else:\n                from huggingface_hub.constants import HF_HUB_CACHE\n                cache = args.cache_dir or Path(HF_HUB_CACHE)\n                phase(\'CHECKING_TRAINING_STORAGE\')\n                storage = check_training_storage(config, cache, output,\n                                                   allow_download=args.allow_model_download)\n                write_json(output / \'storage-check.json\', storage)\n                require(storage[\'passed\'], \'INSUFFICIENT_DISK_FOR_PINNED_BASE_AND_RUN\')\n                phase(\'LOADING_PINNED_BASE\')\n                model = load_training_model(config, allow_download=args.allow_model_download,\n                                             cache_dir=cache)\n                base = model.get_base_model()\n                quantization = base.config.quantization_config\n                write_json(output / \'model-load-report.json\', {\n                    \'model_id\': config[\'model_id\'], \'revision\': base.config._commit_hash,\n                    \'model_class\': type(base).__name__, \'is_loaded_in_4bit\': base.is_loaded_in_4bit,\n                    \'quantization\': quantization.to_dict() if hasattr(quantization, \'to_dict\') else quantization,\n                    \'use_cache\': base.config.use_cache})\n                torch.cuda.reset_peak_memory_stats(0)\n                write_json(output / \'trainable-parameters.json\', parameter_report(model))\n                train_args = training_arguments(config, args.profile, output)\n                write_json(output / \'training-arguments.json\', train_args.to_dict())\n                write_json(output / \'training-device.json\', {\n                    \'visible_gpu_count\': train_args.n_gpu, \'world_size\': train_args.world_size,\n                    \'cuda_visible_devices\': os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n                    \'mode\': \'SINGLE_GPU_NO_DATAPARALLEL\'})\n                phase(\'TRAINING_AND_VALIDATION\')\n                _, update = fit(model, data, collator, train_args, output, config,\n                                 resume=resume, stop_after_step=args.stop_after_step)\n                phase(\'SAVING_ARTIFACTS\')\n                write_json(output / \'vram.json\', {\n                    \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(0),\n                    \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(0)})\n                tokenizer.save_pretrained(output / \'tokenizer\')\n                write_json(output / \'tokenizer-reference.json\', read_json(\n                    args.bundle / \'data/pilot-v2/tokenizer-reference.json\'))\n                status = \'TRAINING_COMPLETED\'\n            phase(status)\n    except BaseException as exc:\n        error = type(exc).__name__\n        phase(\'FAILED\')\n        # Do not print payloads or secrets into public console output.\n        with (output / \'stderr.txt\').open(\'a\', encoding=\'utf-8\') as stream:\n            stream.write(error + \': \' + str(exc) + \'\\n\')\n        raise\n    finally:\n        finish_manifest(output, status=status, profile=args.profile, config=config, error_type=error)\n    print(status + \': \' + str(output))\n    return 0\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n'}
FROZEN_HASHES = {'__init__.py': '0ec7bff3b29cb289cfd2455b6c030b471e3509cb76233276be3a7274f464f5f0', 'check.py': '645cd432489c9285d7b6223610fa2859930b19dcc36309c6641f64538128815b', 'collator.py': '543127c0b8a3e16f0563345e3dcbce7b4e9ae6e882dc2b6ff824566e6517f808', 'data.py': '99d05d205c608eeda28a3ebf2f34c7210305c4ba220cce5e966121e344945262', 'kaggle.py': 'ce19f7dd2eff3e19751ac980e028e62ac079bcd3f17bba63fb3cf09a2246ffcc', 'package.py': 'db4fa3701b1966641804440c98c6bfbc48b4e2cc3f42ae99b1022f3978cdd1ac', 'pilot.py': '420680096328585fb6c23bab5cfb6d502b54281fb5e5230d4c59f90cf0ed29da', 'runtime.py': '311e0899ccb7c8b673dcb60fd4a1d7a9ae4a2ba60b1d103d847f47718b9c2958', 'setup.py': 'cc9810b4414fddabf62757891e59f1daf22c57730bee1dd299d0da68c39f0d8a', 'smoke.py': '2b5a895a13708bb8e91e7a42a95f93d54a02f004c736ca786ac07134016d11c6', 'storage.py': '6df94b551939345b950c211248063c2fa0020a8d2a3c6c7df5b81b7af7834f4c', 'train.py': 'a4a4032bd7bc43859575517585567c486cb43b9ccd6bc4ce0937b66637b8833f'}
FROZEN_TOOLS = WORK / 'tools-pilot-frozen-b228645e1df2'
frozen_files = {'scripts/__init__.py': b'', 'configs/qlora-pilot.json': b'{\n  "schema_version": "ohayo-qlora-recipe-v1",\n  "data_manifest_sha256": "3be5d5f9d70e9bda9b2d0285ffdb78e3186287359ec70f35052b4e1ac3947448",\n  "contract_sha256": "5af1a00516a8ef54631276454a0e905d933a2f04ec53835f407f235ddf8cf99e",\n  "model_id": "Qwen/Qwen2.5-7B-Instruct",\n  "revision": "a09a35458c702b33eeacc393d103063234e8bc28",\n  "seed": 42,\n  "lora": {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05, "bias": "none", "target_modules": "all-linear", "task_type": "CAUSAL_LM"},\n  "training": {"learning_rate": 0.0002, "gradient_accumulation_steps": 4, "per_device_train_batch_size": 1, "per_device_eval_batch_size": 1, "warmup_ratio": 0.1, "max_grad_norm": 1.0, "optim": "paged_adamw_8bit", "lr_scheduler_type": "linear"},\n  "profiles": {"smoke": {"max_steps": 2, "save_steps": 2, "eval_steps": 2}, "pilot": {"max_steps": 50, "save_steps": 25, "eval_steps": 25}},\n  "dependencies": {"transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2", "peft": "0.17.1", "accelerate": "1.10.1", "bitsandbytes": "0.48.2", "safetensors": "0.6.2"}\n}\n'}
for name, content in FROZEN_SOURCES.items():
    assert hashlib.sha256(content.encode()).hexdigest() == FROZEN_HASHES[name]
    frozen_files['scripts/qlora/' + name] = content.encode()
if not FROZEN_TOOLS.exists():
    for name, content in frozen_files.items():
        path = FROZEN_TOOLS / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
assert {p.relative_to(FROZEN_TOOLS).as_posix() for p in FROZEN_TOOLS.rglob('*') if p.is_file()} == set(frozen_files)
for name, content in frozen_files.items():
    assert (FROZEN_TOOLS / name).read_bytes() == content, 'FROZEN_RUNTIME_CHANGED'
VERIFY_DIR = WORK / 'verification' / ('verification-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
BASE_CACHE = WORK / 'base-cache'
VERIFY_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES='0', USE_TF='0', USE_FLAX='0', PYTHONDONTWRITEBYTECODE='1', PYTHONUTF8='1', TOKENIZERS_PARALLELISM='false', HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0', HF_XET_CHUNK_CACHE_SIZE_BYTES='0', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-B', '-m', 'scripts.qlora.assessment', '--stage', 'prepare', '--output', str(VERIFY_DIR), '--work', str(WORK), '--bundle', str(HANDOFF), '--frozen-tools', str(FROZEN_TOOLS)], cwd=VERIFY_TOOLS, env=VERIFY_ENV, check=True)

def run_verified_process(command, cwd, label, phase_path):
    global VERIFY_PROCESS
    if 'VERIFY_PROCESS' in globals() and VERIFY_PROCESS.poll() is None:
        raise RuntimeError('A verification process is already running.')
    stdout_path = VERIFY_DIR / (label + '-stdout.txt')
    stderr_path = VERIFY_DIR / (label + '-stderr.txt')
    with stdout_path.open('x', encoding='utf-8') as stdout, stderr_path.open('x', encoding='utf-8') as stderr:
        VERIFY_PROCESS = subprocess.Popen(command, cwd=cwd, env=VERIFY_ENV, stdout=stdout, stderr=stderr)
        started = time.monotonic()
        try:
            while VERIFY_PROCESS.poll() is None:
                try:
                    phase = json.loads(phase_path.read_text())
                except (FileNotFoundError, ValueError):
                    phase = {}
                print(f"{label}: {int(time.monotonic() - started)}s | {phase.get('phase', 'LOADING_OR_VALIDATING')} | completed={phase.get('completed', '-')} | step={phase.get('step', '-')}", flush=True)
                time.sleep(15)
        except BaseException:
            VERIFY_PROCESS.terminate()
            try:
                VERIFY_PROCESS.wait(timeout=30)
            except subprocess.TimeoutExpired:
                VERIFY_PROCESS.kill()
                VERIFY_PROCESS.wait()
            raise
    if VERIFY_PROCESS.returncode != 0:
        print(stderr_path.read_text(encoding='utf-8')[-5000:])
        raise RuntimeError(label + ' failed. Keep this run and send the error; do not overwrite it.')
    print(stdout_path.read_text(encoding='utf-8')[-2000:])

print('Reload + comparison: 20 cases x 2 arms, plus 2 untimed warmup calls. No training in this cell.')
print('Output:', VERIFY_DIR)
run_verified_process([PYTHON, '-B', '-m', 'scripts.qlora.assessment', '--stage', 'evaluate', '--output', str(VERIFY_DIR), '--cache-dir', str(BASE_CACHE), '--allow-model-download'], VERIFY_TOOLS, 'evaluation', VERIFY_DIR / 'phase.json')


In [ ]:
# Verify checkpoint resume in a NEW process. Exactly one optimizer update: 25 -> 26.
assert (VERIFY_DIR / 'evaluation/comparison.json').is_file(), 'Run reload/comparison first.'
context = json.loads((VERIFY_DIR / 'context.json').read_text())
PILOT_RUN = Path(context['pilot'])
print('Resume verification: 25 -> 26 only. max_steps remains 50 to preserve the original scheduler.')
run_verified_process([PYTHON, '-B', '-m', 'scripts.qlora.train', '--bundle', str(HANDOFF), '--config', str(FROZEN_TOOLS / 'configs/qlora-pilot.json'), '--profile', 'pilot', '--output', str(VERIFY_DIR / 'resume'), '--cache-dir', str(BASE_CACHE), '--allow-model-download', '--resume-from-checkpoint', str(PILOT_RUN / 'checkpoint-25'), '--resume-manifest-sha256', context['pilot_manifest_sha256'], '--stop-after-step', '26'], FROZEN_TOOLS, 'resume', VERIFY_DIR / 'resume/phase.json')


In [ ]:
# CPU artifact audit and private export. Does not run training or generation.
assert VERIFY_PROCESS.returncode == 0 and (VERIFY_DIR / 'resume/run-manifest.json').is_file()
subprocess.run([PYTHON, '-B', '-m', 'scripts.qlora.assessment', '--stage', 'export', '--output', str(VERIFY_DIR)], cwd=VERIFY_TOOLS, env=VERIFY_ENV, check=True)
print(VERIFY_DIR.with_name(VERIFY_DIR.name + '-audit.json').read_text(encoding='utf-8'))
print('Send verification ZIP, audit JSON and receipt. Semantic scores remain blank for content review.')
